# Experiment: DINO+SAM2 2.5D ByteTrack Crop Extraction

Notebook-level feasibility experiment for an upgraded training-crop extraction pipeline.

This is not a GUI implementation and not a production refactor. It keeps the old DINO+SAM2 proposal/crop format as the starting point, then tests whether monocular-depth 2.5D anchors plus ByteTrack-style association can produce more reliable track-level crop exports.

Key principle: **tracking identity and semantic label are decoupled**. The tracker may keep a long object identity even if raw DINO labels flip; final export labels are chosen by track-level smoothing and coarse voting.

## 1. Setup and Config

In [ ]:
from __future__ import annotations

import ast
import json
import math
import os
import random
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    import torch
except Exception:
    torch = None

try:
    import yaml
except Exception:
    yaml = None

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "auto_label" / "generate_mask_proposals.py").exists():
            return p
    return start


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def auto_device(requested: str = "cuda") -> str:
    if requested == "cuda" and torch is not None and torch.cuda.is_available():
        return "cuda"
    return "cpu"


def repo_path(value: str | Path | None) -> Path | None:
    if value is None or str(value).strip() == "":
        return None
    p = Path(value)
    if p.is_absolute():
        return p
    return REPO_ROOT / p


def find_egtea_video() -> str:
    roots = [
        Path(r"C:\Users\18447\Detector\data\egtea_gaze_plus\video_clips\cropped_clips"),
        REPO_ROOT.parent / "data" / "egtea_gaze_plus" / "video_clips" / "cropped_clips",
        REPO_ROOT / "data" / "egtea_gaze_plus" / "video_clips" / "cropped_clips",
    ]
    exts = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts])
    if not candidates:
        return "path/to/long_video.mp4"
    candidates = sorted(candidates, key=lambda p: p.stat().st_size, reverse=True)
    return str(candidates[0])


def load_old_auto_label_defaults() -> dict[str, Any]:
    cfg_path = REPO_ROOT / "configs" / "auto_label_demo.yaml"
    defaults = {
        "prompts": [
            "hand", "glove", "pot", "pan", "lid", "cookware", "tray", "kettle",
            "bowl", "plate", "cup", "glass", "bottle", "jar",
            "container", "box", "package", "bag", "carton", "can",
            "knife", "fork", "spoon", "spatula", "tongs", "ladle", "whisk", "peeler", "scissors", "cutting board",
            "pasta", "noodles", "rice", "bread", "vegetable", "fruit", "meat", "fish", "egg", "cheese",
            "ingredient", "food", "dry food", "liquid", "water", "milk", "sauce", "oil", "powder", "sugar", "salt",
            "sink", "faucet", "stove", "cooktop", "oven", "microwave", "fridge", "drawer", "cabinet", "countertop",
            "table", "rack", "sponge", "towel",
        ],
        "confidence_threshold": 0.25,
        "box_threshold": 0.25,
        "text_threshold": 0.20,
        "max_objects_per_frame": 40,
        "sam2_checkpoint": "models/sam2/sam2_hiera_tiny.pt",
        "sam2_config": "models/sam2/sam2_hiera_t.yaml",
        "detector_model_path": "models/groundingdino_swint_ogc.pth",
    }
    if cfg_path.exists() and yaml is not None:
        cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}
        prop = cfg.get("proposal_generation", {})
        defaults.update({
            "prompts": prop.get("prompts") or defaults["prompts"],
            "confidence_threshold": float(prop.get("confidence_threshold", defaults["confidence_threshold"])),
            "box_threshold": float(prop.get("box_threshold", defaults["box_threshold"])),
            "text_threshold": float(prop.get("text_threshold", defaults["text_threshold"])),
            "max_objects_per_frame": int(prop.get("max_objects_per_frame", defaults["max_objects_per_frame"])),
            "sam2_checkpoint": prop.get("sam2_checkpoint", defaults["sam2_checkpoint"]),
            "sam2_config": prop.get("sam2_config", defaults["sam2_config"]),
            "detector_model_path": prop.get("detector_model_path", defaults["detector_model_path"]),
        })
    return defaults


OLD_DEFAULTS = load_old_auto_label_defaults()

CONFIG = {
    "VIDEO_PATH": find_egtea_video(),
    "OUTPUT_DIR": "outputs/exp_2p5d_bytetrack_crop_extraction",

    "FRAME_STRIDE": 5,
    "MAX_FRAMES": None,  # set e.g. 150 for quick debugging
    "START_FRAME": 0,
    "REUSE_EXTRACTED_FRAMES": True,  # if frames_metadata.json exists, skip frame extraction on Run All
    "REUSE_NORMALIZED_PROPOSALS": True,
    "REUSE_ENRICHED_DETECTIONS": True,
    "REUSE_TRACKS_RAW": True,
    "REUSE_TRACKING_VISUALIZATION_VIDEO": True,
    "REUSE_BASELINE_COMPARISON": True,
    "REUSE_SCORED_TRACKS": False,  # scoring is cheap; keep False so config changes apply
    "REUSE_DATASET_EXPORT": False,  # export is cheap; keep False so filters apply
    "SKIP_HEAVY_STEPS_IF_CACHE_EXISTS": True,  # Run All should reuse frames/proposals/depth/embeddings/tracks when present

    "DEVICE": "cuda",

    # DINO+SAM2 proposal settings
    "USE_EXISTING_DINO_SAM2_RESULTS": True,  # temporarily reuse completed DINO+SAM2 proposals
    "EXISTING_RESULTS_PATH": "outputs/exp_2p5d_bytetrack_crop_extraction/dino_sam2_raw/proposals.jsonl",
    "DINO_SAM2_PROMPT": None,
    "DINO_BOX_THRESHOLD": None,
    "DINO_TEXT_THRESHOLD": None,
    "DINO_CONFIDENCE_THRESHOLD": None,
    "MAX_OBJECTS_PER_FRAME": None,
    "SAM2_CONFIG": None,
    "SAM2_CHECKPOINT": None,
    "DETECTOR_MODEL_PATH": None,

    # Depth / 2.5D
    "DEPTH_MODEL": "depth_anything_v2",  # depth_anything_v2, midas, dummy
    "DEPTH_ANYTHING_REPO": r"C:\tmp\Depth-Anything-V2",
    "DEPTH_ANYTHING_ENCODER": "vits",  # vits/vitb/vitl; vits is fastest for experiments
    "DEPTH_ANYTHING_CHECKPOINT": "models/depth_anything_v2_vits.pth",
    "DEPTH_INVERT": False,
    "DEPTH_EVERY_N_EXTRACTED_FRAMES": 1,
    "MIN_MASK_AREA": 100,
    "MIN_DEPTH_PIXELS": 50,

    # ByteTrack-style tracking
    "HIGH_CONF_THRESHOLD": 0.45,
    "LOW_CONF_THRESHOLD": 0.15,
    "MATCH_THRESHOLD_STAGE1": 0.35,
    "MATCH_THRESHOLD_STAGE2": 0.25,
    "MAX_MISSING_FRAMES": 8,
    "MAX_OCCLUDED_FRAMES": 30,
    "EMA_ALPHA": 0.7,

    # Association weights
    "W_IOU": 0.25,
    "W_CENTER": 0.15,
    "W_DEPTH": 0.20,
    "W_MASK": 0.15,
    "W_APPEARANCE": 0.20,
    "W_LABEL": 0.05,

    # Crop candidate/export settings
    "MAX_CANDIDATES_PER_TRACK": 40,
    "EXPORT_MIN_PER_TRACK": 1,
    "EXPORT_MAX_PER_TRACK": 8,
    "IMPORTANT_CLASS_EXPORT_MAX": 15,
    "MIN_TRACK_LENGTH_FOR_SILVER": 5,
    "MIN_TRACK_QUALITY_FOR_SILVER": 0.65,
    "MIN_TRACK_QUALITY_FOR_GOLD_CANDIDATE": 0.80,

    # Review logic
    "LABEL_CONFIDENCE_REVIEW_THRESHOLD": 0.65,
    "RAW_LABEL_ENTROPY_REVIEW_THRESHOLD": 1.2,
    "APPEARANCE_CONSISTENCY_REVIEW_THRESHOLD": 0.55,
    "DEPTH_STABILITY_REVIEW_THRESHOLD": 0.55,
    "MASK_QUALITY_REVIEW_THRESHOLD": 0.60,

    # Output
    "SAVE_VISUALIZATION_VIDEO": True,
    "SAVE_CONTACT_SHEETS": True,
    "SAVE_DEBUG_JSON": True,
    "SAVE_TRACK_CSV": True,
}

CONFIG.update({
    # Appearance embedding
    "APPEARANCE_MODEL": "dinov2",
    "USE_MASKED_CROP_FOR_EMBEDDING": True,
    "MASK_BACKGROUND_MODE": "gray",
    "EMBEDDING_BATCH_SIZE": 16,
    "EMBEDDING_CACHE_DIR": "outputs/exp_2p5d_bytetrack_crop_extraction/debug/embeddings",

    # Appearance-heavy association
    "W_APPEARANCE": 0.30,
    "W_IOU": 0.20,
    "W_CENTER": 0.10,
    "W_DEPTH": 0.15,
    "W_MASK": 0.10,
    "W_SIZE_SHAPE": 0.10,
    "W_LABEL": 0.03,
    "W_MOTION": 0.02,

    # Hard anti-merge gates
    "ENABLE_HARD_GATES": True,
    "MIN_APPEARANCE_FOR_MATCH": 0.35,
    "MIN_APPEARANCE_IN_CONFLICT": 0.50,
    "MAX_SIZE_CHANGE_RATIO": 2.5,
    "MAX_DEPTH_DIFF_FOR_LOW_IOU": 0.25,
    "MIN_IOU_TO_IGNORE_DEPTH_GATE": 0.30,

    # Conflict zone
    "ENABLE_CONFLICT_ZONE": True,
    "CONFLICT_PAIRWISE_IOU_THRESHOLD": 0.25,
    "CONFLICT_CENTER_DISTANCE_THRESHOLD": 0.08,
    "CONFLICT_DEPTH_DIFF_THRESHOLD": 0.10,
    "CONFLICT_WEAK_UPDATE_ALPHA": 0.90,
    "DO_NOT_UPDATE_GALLERY_IN_CONFLICT": True,
    "DO_NOT_EXPORT_CONFLICT_FRAMES": True,

    # Track appearance gallery
    "GALLERY_MAX_BEST": 10,
    "GALLERY_MAX_RECENT": 5,
    "GALLERY_MIN_QUALITY_TO_ADD": 0.65,
    "GALLERY_MIN_APPEARANCE_TO_EXISTING": 0.45,
    "GALLERY_UPDATE_ONLY_VISIBLE": True,

    # Matching strategy
    "MATCHING_STRATEGY": "hungarian",
    "USE_HUNGARIAN_IN_CONFLICT_ONLY": False,

    # Track split detection
    "ENABLE_TRACK_SPLIT_DETECTION": True,
    "SPLIT_MIN_TRACK_EMBEDDINGS": 8,
    "SPLIT_MAX_INTRA_VARIANCE": 0.35,
    "SPLIT_MIN_TWO_CLUSTER_SEPARATION": 0.25,
    "SPLIT_MARK_REVIEW_ONLY": True,

    # Export filters
    "EXPORT_REQUIRE_APPEARANCE_CONSISTENT": True,
    "MIN_TRACK_APPEARANCE_CONSISTENCY_FOR_EXPORT": 0.60,
    "REJECT_SPLIT_CANDIDATES_FROM_GOLD_SILVER": True,
})

CONFIG["DEVICE"] = auto_device(CONFIG["DEVICE"])
OUTPUT_DIR = repo_path(CONFIG["OUTPUT_DIR"])
assert OUTPUT_DIR is not None

DIRS = {
    "frames": OUTPUT_DIR / "frames",
    "dino_raw": OUTPUT_DIR / "dino_sam2_raw",
    "depth": OUTPUT_DIR / "depth",
    "crops_all": OUTPUT_DIR / "crops_all",
    "masks_all": OUTPUT_DIR / "masks_all",
    "tracks": OUTPUT_DIR / "tracks",
    "review_contact_sheets": OUTPUT_DIR / "review_contact_sheets",
    "debug": OUTPUT_DIR / "debug",
    "visualizations": OUTPUT_DIR / "visualizations",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)
EMBEDDING_CACHE_DIR = repo_path(CONFIG.get("EMBEDDING_CACHE_DIR")) or (OUTPUT_DIR / "debug" / "embeddings")
CONFIG["EMBEDDING_CACHE_DIR"] = str(EMBEDDING_CACHE_DIR)
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)
for split in ["gold", "silver", "review", "reject"]:
    (OUTPUT_DIR / "export" / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "export" / split / "masks").mkdir(parents=True, exist_ok=True)

resolved = dict(CONFIG)
resolved["DINO_SAM2_PROMPT_RESOLVED"] = CONFIG["DINO_SAM2_PROMPT"] or ", ".join(OLD_DEFAULTS["prompts"])
resolved["DINO_BOX_THRESHOLD_RESOLVED"] = CONFIG["DINO_BOX_THRESHOLD"] if CONFIG["DINO_BOX_THRESHOLD"] is not None else OLD_DEFAULTS["box_threshold"]
resolved["DINO_TEXT_THRESHOLD_RESOLVED"] = CONFIG["DINO_TEXT_THRESHOLD"] if CONFIG["DINO_TEXT_THRESHOLD"] is not None else OLD_DEFAULTS["text_threshold"]
resolved["SAM2_CHECKPOINT_RESOLVED"] = CONFIG["SAM2_CHECKPOINT"] or OLD_DEFAULTS["sam2_checkpoint"]
resolved["SAM2_CONFIG_RESOLVED"] = CONFIG["SAM2_CONFIG"] or OLD_DEFAULTS["sam2_config"]

print("repo root:", REPO_ROOT)
print("device:", CONFIG["DEVICE"])
print(json.dumps(resolved, indent=2, ensure_ascii=False))

## 2. Extract Frames from One Long Video

In [ ]:
def load_existing_frames_metadata() -> list[dict[str, Any]] | None:
    metadata_path = OUTPUT_DIR / "frames_metadata.json"
    if not metadata_path.exists():
        return None
    try:
        records = json.loads(metadata_path.read_text(encoding="utf-8"))
    except Exception as exc:
        print("Could not read existing frames_metadata.json:", exc)
        return None
    valid = []
    for rec in records:
        frame_path = Path(str(rec.get("frame_path", "")))
        if not frame_path.is_absolute():
            frame_path = REPO_ROOT / frame_path
        if frame_path.exists():
            rec = dict(rec)
            rec["frame_path"] = str(frame_path)
            valid.append(rec)
    if not valid:
        return None
    if CONFIG["MAX_FRAMES"] is not None:
        valid = valid[: int(CONFIG["MAX_FRAMES"])]
    print(f"Reusing existing extracted frames: {len(valid)} from {metadata_path}")
    return valid


def extract_frames_from_video(config: dict[str, Any]) -> list[dict[str, Any]]:
    if config.get("REUSE_EXTRACTED_FRAMES", True):
        existing = load_existing_frames_metadata()
        if existing is not None:
            return existing

    video_path = repo_path(config["VIDEO_PATH"])
    if video_path is None or not video_path.exists():
        raise FileNotFoundError(f"Set CONFIG['VIDEO_PATH'] to an EGTEA video. Current: {config['VIDEO_PATH']}")
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    stride = int(config["FRAME_STRIDE"])
    start = int(config["START_FRAME"])
    max_frames = config["MAX_FRAMES"]
    records = []
    extracted_idx = 0
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    original_idx = start
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if original_idx >= start and (original_idx - start) % stride == 0:
                out = DIRS["frames"] / f"frame_{extracted_idx:06d}.jpg"
                cv2.imwrite(str(out), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
                records.append({
                    "extracted_frame_idx": extracted_idx,
                    "original_frame_idx": original_idx,
                    "frame_index": original_idx,
                    "timestamp_sec": (original_idx / fps) if fps else 0.0,
                    "timestamp": (original_idx / fps) if fps else 0.0,
                    "frame_path": str(out),
                    "video_path": str(video_path),
                })
                extracted_idx += 1
                if max_frames is not None and extracted_idx >= int(max_frames):
                    break
            original_idx += 1
    finally:
        cap.release()
    (OUTPUT_DIR / "frames_metadata.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
    print("video:", video_path)
    print("total video frames:", total)
    print("FPS:", fps)
    print("extracted frames:", len(records))
    print("effective extracted FPS:", fps / max(1, stride) if fps else 0)
    return records


frames_metadata = extract_frames_from_video(CONFIG)

sample = frames_metadata[:5]
if sample:
    fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
    if len(sample) == 1:
        axes = [axes]
    for ax, rec in zip(axes, sample):
        img_bgr = cv2.imread(rec["frame_path"])
        if img_bgr is not None:
            img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        ax.set_title(f"ext {rec['extracted_frame_idx']} / orig {rec['original_frame_idx']}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 3. Run or Load Existing DINO+SAM2 Proposals

In [ ]:
def prompt_list_from_config() -> list[str]:
    if CONFIG["DINO_SAM2_PROMPT"]:
        return [p.strip() for p in str(CONFIG["DINO_SAM2_PROMPT"]).split(",") if p.strip()]
    return list(OLD_DEFAULTS["prompts"])


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def parse_bbox(row: dict[str, Any]) -> list[float] | None:
    for key in ["bbox", "bbox_xyxy"]:
        value = row.get(key)
        if isinstance(value, str):
            try:
                value = json.loads(value)
            except Exception:
                try:
                    value = ast.literal_eval(value)
                except Exception:
                    value = None
        if isinstance(value, (list, tuple)) and len(value) >= 4:
            return [float(x) for x in value[:4]]
    if all(k in row for k in ["x1", "y1", "x2", "y2"]):
        return [float(row["x1"]), float(row["y1"]), float(row["x2"]), float(row["y2"])]
    return None


def resolve_path(value) -> Path | None:
    if value is None or str(value).strip() == "":
        return None
    p = Path(str(value))
    if p.exists():
        return p
    if not p.is_absolute():
        q = REPO_ROOT / p
        if q.exists():
            return q
    return p


def bbox_fallback_mask(mask_path: Path, bbox: list[float], shape_hw: tuple[int, int]) -> None:
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
    if x2 > x1 and y2 > y1:
        mask[y1:y2, x1:x2] = 255
    cv2.imwrite(str(mask_path), mask)


def save_crop(frame_path: Path, bbox: list[float], mask_path: Path | None, out_path: Path) -> None:
    img = cv2.imread(str(frame_path))
    if img is None:
        return
    h, w = img.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
    crop = img[y1:y2, x1:x2].copy()
    if crop.size == 0:
        return
    if mask_path and mask_path.exists():
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            m = mask[y1:y2, x1:x2] > 0
            if m.shape[:2] == crop.shape[:2]:
                bg = np.full_like(crop, 114)
                crop = np.where(m[:, :, None], crop, bg)
    cv2.imwrite(str(out_path), crop, [cv2.IMWRITE_JPEG_QUALITY, 90])


def normalize_proposal_records(rows: list[dict[str, Any]], frames_metadata: list[dict[str, Any]]) -> list[dict[str, Any]]:
    frame_by_path = {str(Path(r["frame_path"]).resolve()): r for r in frames_metadata}
    frame_by_stem = {Path(r["frame_path"]).stem: r for r in frames_metadata}
    grouped: dict[int, dict[str, Any]] = {}
    for i, row in enumerate(rows):
        frame_path = resolve_path(row.get("frame_path") or row.get("image_path"))
        frame_rec = frame_by_path.get(str(frame_path.resolve())) if frame_path and frame_path.exists() else None
        if frame_rec is None and frame_path is not None:
            frame_rec = frame_by_stem.get(frame_path.stem)
        if frame_rec is None:
            continue
        ext_idx = int(frame_rec["extracted_frame_idx"])
        bbox = parse_bbox(row)
        if bbox is None:
            continue
        det_id = str(row.get("proposal_id", row.get("det_id", i)))
        label = str(row.get("raw_label") or row.get("dino_label") or row.get("label") or row.get("predicted_label") or "unknown")
        conf = float(row.get("conf") or row.get("confidence") or row.get("score") or 0.0)

        mask_src = resolve_path(row.get("mask_path"))
        crop_src = resolve_path(row.get("crop_path"))
        mask_out = DIRS["masks_all"] / f"det_{int(ext_idx):06d}_{len(grouped.get(ext_idx, {}).get('detections', [])):04d}.png"
        crop_out = DIRS["crops_all"] / f"det_{int(ext_idx):06d}_{len(grouped.get(ext_idx, {}).get('detections', [])):04d}.jpg"

        mask_source = "sam2"
        if mask_src and mask_src.exists():
            shutil.copy2(mask_src, mask_out)
        else:
            img = cv2.imread(frame_rec["frame_path"])
            if img is None:
                continue
            bbox_fallback_mask(mask_out, bbox, img.shape[:2])
            mask_source = "bbox_fallback"

        if crop_src and crop_src.exists():
            shutil.copy2(crop_src, crop_out)
        else:
            save_crop(Path(frame_rec["frame_path"]), bbox, mask_out, crop_out)

        det = {
            "det_id": det_id,
            "raw_label": label,
            "dino_label": row.get("dino_label") or row.get("predicted_label") or row.get("label") or label,
            "cluster_label": row.get("cluster_label", ""),
            "conf": conf,
            "bbox": bbox,
            "mask_path": str(mask_out),
            "crop_path": str(crop_out),
            "mask_source": mask_source,
            "raw_metadata": row,
        }
        grouped.setdefault(ext_idx, {
            "extracted_frame_idx": ext_idx,
            "original_frame_idx": int(frame_rec["original_frame_idx"]),
            "frame_path": frame_rec["frame_path"],
            "detections": [],
        })["detections"].append(det)
    return [grouped.get(i, {
        "extracted_frame_idx": int(r["extracted_frame_idx"]),
        "original_frame_idx": int(r["original_frame_idx"]),
        "frame_path": r["frame_path"],
        "detections": [],
    }) for i, r in enumerate(frames_metadata)]


def run_old_dino_sam2_script(frames_metadata: list[dict[str, Any]]) -> Path:
    metadata_path = OUTPUT_DIR / "frames_metadata.json"
    prompt = ",".join(prompt_list_from_config())
    cmd = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "auto_label" / "generate_mask_proposals.py"),
        "--frames-root", str(DIRS["frames"]),
        "--metadata", str(metadata_path),
        "--output", str(DIRS["dino_raw"]),
        "--backend", "groundingdino_sam2",
        "--prompts", prompt,
        "--device", CONFIG["DEVICE"],
        "--confidence", str(CONFIG["DINO_CONFIDENCE_THRESHOLD"] if CONFIG["DINO_CONFIDENCE_THRESHOLD"] is not None else OLD_DEFAULTS["confidence_threshold"]),
        "--box-threshold", str(CONFIG["DINO_BOX_THRESHOLD"] if CONFIG["DINO_BOX_THRESHOLD"] is not None else OLD_DEFAULTS["box_threshold"]),
        "--text-threshold", str(CONFIG["DINO_TEXT_THRESHOLD"] if CONFIG["DINO_TEXT_THRESHOLD"] is not None else OLD_DEFAULTS["text_threshold"]),
        "--max-objects-per-frame", str(CONFIG["MAX_OBJECTS_PER_FRAME"] if CONFIG["MAX_OBJECTS_PER_FRAME"] is not None else OLD_DEFAULTS["max_objects_per_frame"]),
        "--detector-model-path", str(repo_path(CONFIG["DETECTOR_MODEL_PATH"] or OLD_DEFAULTS["detector_model_path"])),
        "--sam2-checkpoint", str(repo_path(CONFIG["SAM2_CHECKPOINT"] or OLD_DEFAULTS["sam2_checkpoint"])),
        "--sam2-config", str(repo_path(CONFIG["SAM2_CONFIG"] or OLD_DEFAULTS["sam2_config"])),
        "--save-debug-vis",
        "--debug-vis-limit", "30",
    ]
    print("Running old DINO+SAM2 extraction command:")
    print(" ".join(f'"{c}"' if " " in str(c) else str(c) for c in cmd))
    result = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True, capture_output=True)
    stdout = result.stdout or ""
    stderr = result.stderr or ""
    (DIRS["debug"] / "dino_sam2_stdout.txt").write_text(stdout, encoding="utf-8")
    (DIRS["debug"] / "dino_sam2_stderr.txt").write_text(stderr, encoding="utf-8")
    print(stdout[-3000:])
    if result.returncode != 0:
        print(stderr[-3000:])
        raise RuntimeError(f"DINO+SAM2 proposal script failed with exit code {result.returncode}")
    return DIRS["dino_raw"] / "proposals.jsonl"


def run_or_load_dino_sam2_proposals(frames_metadata: list[dict[str, Any]], config: dict[str, Any]) -> list[dict[str, Any]]:
    normalized_cache = DIRS["dino_raw"] / "proposals_normalized.json"
    if config.get("REUSE_NORMALIZED_PROPOSALS", True) and normalized_cache.exists():
        print(f"Reusing normalized proposals: {normalized_cache}")
        normalized = json.loads(normalized_cache.read_text(encoding="utf-8"))
        flat = [d for fr in normalized for d in fr.get("detections", [])]
        print("frames:", len(normalized))
        print("frames with detections:", sum(1 for fr in normalized if fr.get("detections")))
        print("total detections:", len(flat))
        print("labels:", Counter(d.get("raw_label", "") for d in flat).most_common(30))
        print("mask sources:", Counter(d.get("mask_source", "") for d in flat))
        return normalized

    if config["USE_EXISTING_DINO_SAM2_RESULTS"]:
        path = repo_path(config["EXISTING_RESULTS_PATH"])
        if path is None or not path.exists():
            raise FileNotFoundError("EXISTING_RESULTS_PATH is enabled but missing.")
    else:
        path = run_old_dino_sam2_script(frames_metadata)
    if path.suffix.lower() == ".jsonl":
        rows = load_jsonl(path)
    elif path.suffix.lower() == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        rows = data if isinstance(data, list) else data.get("detections", data.get("proposals", []))
    elif path.suffix.lower() == ".csv":
        rows = pd.read_csv(path).to_dict(orient="records")
    elif path.suffix.lower() == ".parquet":
        rows = pd.read_parquet(path).to_dict(orient="records")
    else:
        raise ValueError(f"Unsupported result path: {path}")

    normalized = normalize_proposal_records(rows, frames_metadata)
    out = DIRS["dino_raw"] / "proposals_normalized.json"
    out.write_text(json.dumps(normalized, indent=2, default=str), encoding="utf-8")

    flat = [d for fr in normalized for d in fr["detections"]]
    print("frames:", len(normalized))
    print("frames with detections:", sum(1 for fr in normalized if fr["detections"]))
    print("total detections:", len(flat))
    print("labels:", Counter(d["raw_label"] for d in flat).most_common(30))
    print("mask sources:", Counter(d["mask_source"] for d in flat))
    return normalized


proposals_by_frame = run_or_load_dino_sam2_proposals(frames_metadata, CONFIG)


def overlay_frame(frame_rec: dict[str, Any], alpha=0.35):
    img = cv2.imread(frame_rec["frame_path"])
    if img is None:
        return None
    out = img.copy()
    rng = np.random.default_rng(123)
    for det in frame_rec["detections"]:
        color = rng.integers(40, 255, size=3).tolist()
        mask = cv2.imread(det["mask_path"], cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            m = mask > 0
            out[m] = (out[m] * (1 - alpha) + np.array(color) * alpha).astype(np.uint8)
        x1, y1, x2, y2 = [int(v) for v in det["bbox"]]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        cv2.putText(out, f"{det['raw_label']} {det['conf']:.2f}", (x1, max(16, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)


sample_frames = [fr for fr in proposals_by_frame if fr["detections"]][:4]
if sample_frames:
    fig, axes = plt.subplots(1, len(sample_frames), figsize=(5 * len(sample_frames), 4))
    if len(sample_frames) == 1:
        axes = [axes]
    for ax, fr in zip(axes, sample_frames):
        ax.imshow(overlay_frame(fr))
        ax.set_title(f"frame {fr['extracted_frame_idx']} dets {len(fr['detections'])}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 4. Depth / 2.5D Enrichment

In [ ]:

class DepthWrapper:
    def __init__(self, mode: str, device: str, invert: bool):
        self.mode = mode
        self.device = device
        self.invert = invert
        self.model = None
        self.transform = None
        self.available_mode = mode
        if mode == "depth_anything_v2":
            self._try_depth_anything()
        elif mode == "midas":
            self._try_midas()

    def _try_depth_anything(self):
        try:
            repo = Path(CONFIG.get("DEPTH_ANYTHING_REPO", ""))
            if repo.exists() and str(repo) not in sys.path:
                sys.path.insert(0, str(repo))
            from depth_anything_v2.dpt import DepthAnythingV2
            if torch is None:
                raise RuntimeError("torch unavailable")
            model_configs = {
                "vits": {"encoder": "vits", "features": 64, "out_channels": [48, 96, 192, 384]},
                "vitb": {"encoder": "vitb", "features": 128, "out_channels": [96, 192, 384, 768]},
                "vitl": {"encoder": "vitl", "features": 256, "out_channels": [256, 512, 1024, 1024]},
            }
            encoder = CONFIG.get("DEPTH_ANYTHING_ENCODER", "vits")
            self.model = DepthAnythingV2(**model_configs[encoder])
            ckpt = repo_path(CONFIG.get("DEPTH_ANYTHING_CHECKPOINT"))
            if ckpt is None or not ckpt.exists():
                raise FileNotFoundError(f"Depth Anything checkpoint not found: {ckpt}")
            self.model.load_state_dict(torch.load(str(ckpt), map_location="cpu"))
            self.model = self.model.to(self.device).eval()
            self.available_mode = f"depth_anything_v2_{encoder}"
            print(f"Depth Anything V2 ready: encoder={encoder}, checkpoint={ckpt}")
        except Exception as exc:
            print("Depth Anything V2 unavailable, falling back to dummy:", exc)
            self.available_mode = "dummy"
            self.model = None

    def _try_midas(self):
        try:
            if torch is None:
                raise RuntimeError("torch unavailable")
            self.model = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
            self.model.to(self.device).eval()
            transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
            self.transform = transforms.small_transform
        except Exception as exc:
            print("MiDaS unavailable, falling back to dummy:", exc)
            self.available_mode = "dummy"
            self.model = None

    def predict(self, rgb: np.ndarray) -> np.ndarray:
        h, w = rgb.shape[:2]
        if self.available_mode == "dummy" or self.model is None:
            gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
            y = np.linspace(0, 1, h, dtype=np.float32)[:, None]
            depth = 0.65 * y + 0.35 * (1.0 - gray)
        elif self.available_mode == "midas":
            inp = self.transform(rgb).to(self.device)
            with torch.inference_mode():
                pred = self.model(inp)
                pred = torch.nn.functional.interpolate(pred.unsqueeze(1), size=(h, w), mode="bicubic", align_corners=False).squeeze()
            depth = pred.detach().cpu().numpy().astype(np.float32)
        else:
            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            with torch.inference_mode():
                depth = self.model.infer_image(bgr)
            depth = cv2.resize(depth.astype(np.float32), (w, h))
        depth = depth.astype(np.float32)
        depth -= np.nanmin(depth)
        depth /= max(1e-6, np.nanmax(depth))
        if self.invert:
            depth = 1.0 - depth
        return depth


def depth_cache_stem(idx: int, wrapper: DepthWrapper) -> str:
    mode = str(wrapper.available_mode).replace("/", "_").replace("\\", "_")
    inv = "inv1" if CONFIG.get("DEPTH_INVERT") else "inv0"
    return f"depth_{mode}_{inv}_{idx:06d}"


def load_or_compute_depth(frame_rec: dict[str, Any], wrapper: DepthWrapper) -> np.ndarray:
    idx = int(frame_rec["extracted_frame_idx"])
    stem = depth_cache_stem(idx, wrapper)
    npy = DIRS["depth"] / f"{stem}.npy"
    png = DIRS["depth"] / f"{stem}.png"
    if npy.exists():
        return np.load(npy)
    rgb = cv2.cvtColor(cv2.imread(frame_rec["frame_path"]), cv2.COLOR_BGR2RGB)
    depth = wrapper.predict(rgb)
    np.save(npy, depth)
    vis = (plt.cm.viridis(depth)[:, :, :3] * 255).astype(np.uint8)
    cv2.imwrite(str(png), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    return depth


def l2_normalize(vec):
    if vec is None:
        return None
    arr = np.asarray(vec, dtype=np.float32).reshape(-1)
    norm = float(np.linalg.norm(arr))
    if norm <= 1e-8:
        return arr.tolist()
    return (arr / norm).tolist()


def vector_or_none(x):
    if x is None:
        return None
    arr = np.asarray(x, dtype=np.float32).reshape(-1)
    if arr.size == 0 or not np.all(np.isfinite(arr)):
        return None
    return arr


def cosine_similarity(a, b, default=0.5):
    aa = vector_or_none(a)
    bb = vector_or_none(b)
    if aa is None or bb is None:
        return default
    den = float(np.linalg.norm(aa) * np.linalg.norm(bb))
    if den <= 1e-8:
        return default
    return float(np.clip(np.dot(aa, bb) / den, -1, 1))


def make_masked_crop(frame_bgr, bbox, mask, mode="gray", padding=0):
    h, w = frame_bgr.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, x1 - padding); y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding); y2 = min(h, y2 + padding)
    crop = frame_bgr[y1:y2, x1:x2].copy()
    if crop.size == 0:
        return None, "empty"
    if mask is None or mask.size == 0:
        return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), "bbox_fallback"
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    m = mask[y1:y2, x1:x2] > 0
    if m.shape[:2] != crop.shape[:2] or not np.any(m):
        return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), "bbox_fallback"
    if not CONFIG.get("USE_MASKED_CROP_FOR_EMBEDDING", True):
        return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), "bbox_crop"
    out = crop.copy()
    if mode == "black":
        out[~m] = 0
    elif mode == "mean_color":
        out[~m] = np.mean(crop[m], axis=0).astype(np.uint8)
    elif mode == "blur":
        blurred = cv2.GaussianBlur(crop, (21, 21), 0)
        out = blurred
        out[m] = crop[m]
    else:
        out[~m] = 127
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB), "sam2_masked"


def color_hist_rgb(crop_rgb: np.ndarray) -> list[float]:
    if crop_rgb is None or crop_rgb.size == 0:
        return []
    hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [12, 6, 4], [0, 180, 0, 256, 0, 256]).flatten().astype(np.float32)
    return l2_normalize(hist)


class AppearanceEmbedder:
    def __init__(self, model_name: str, device: str):
        self.requested = model_name
        self.model_name = model_name
        self.device = device
        self.model = None
        self.processor = None
        self.transform = None
        self.cache_dir = repo_path(CONFIG["EMBEDDING_CACHE_DIR"])
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        if model_name == "none":
            return
        if model_name == "dinov2":
            self._try_dinov2()
        elif model_name in {"clip", "siglip"}:
            self._try_transformers(model_name)
        if self.model is None and model_name not in {"color_hist", "none"}:
            print(f"Appearance model {model_name} unavailable; falling back to color_hist")
            self.model_name = "color_hist"

    def _try_dinov2(self):
        try:
            if torch is None:
                raise RuntimeError("torch unavailable")
            self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
            self.model.to(self.device).eval()
            import torchvision.transforms as T
            self.transform = T.Compose([
                T.ToPILImage(), T.Resize((224, 224)), T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
            self.model_name = "dinov2_vits14"
            print("DINOv2 appearance embedder ready")
        except Exception as exc:
            print("DINOv2 unavailable:", exc)
            self.model = None

    def _try_transformers(self, model_name):
        try:
            from transformers import AutoImageProcessor, AutoModel, AutoProcessor
            if model_name == "clip":
                mid = "openai/clip-vit-base-patch32"
                self.processor = AutoProcessor.from_pretrained(mid, local_files_only=True)
            else:
                mid = "google/siglip-base-patch16-224"
                self.processor = AutoImageProcessor.from_pretrained(mid, local_files_only=True)
            self.model = AutoModel.from_pretrained(mid, local_files_only=True).to(self.device).eval()
            self.model_name = model_name
        except Exception as exc:
            print(f"{model_name} unavailable:", exc)
            self.model = None

    def cache_path(self, crop_path, mask_path, masked_source):
        key = f"{Path(str(crop_path)).stem}_{Path(str(mask_path)).stem}_{self.model_name}_{CONFIG['MASK_BACKGROUND_MODE']}_{masked_source}"
        return self.cache_dir / f"{key}.npy"

    def embed(self, crop_rgb, crop_path="", mask_path="", masked_source=""):
        if self.model_name == "none":
            return None, None
        cache = self.cache_path(crop_path, mask_path, masked_source)
        if cache.exists():
            return np.load(cache).astype(np.float32).tolist(), str(cache)
        try:
            if self.model_name.startswith("dinov2") and self.model is not None:
                x = self.transform(crop_rgb).unsqueeze(0).to(self.device)
                with torch.inference_mode():
                    emb = self.model(x).detach().cpu().numpy()[0]
                vec = l2_normalize(emb)
            elif self.model is not None and self.processor is not None:
                from PIL import Image
                inputs = self.processor(images=Image.fromarray(crop_rgb), return_tensors="pt").to(self.device)
                with torch.inference_mode():
                    out = self.model.get_image_features(**inputs) if hasattr(self.model, "get_image_features") else self.model(**inputs).pooler_output
                vec = l2_normalize(out.detach().cpu().numpy()[0])
            else:
                vec = color_hist_rgb(crop_rgb)
            np.save(cache, np.asarray(vec, dtype=np.float32))
            return vec, str(cache)
        except Exception as exc:
            print("embedding failed, using color_hist fallback:", exc)
            vec = color_hist_rgb(crop_rgb)
            np.save(cache, np.asarray(vec, dtype=np.float32))
            return vec, str(cache)


def connected_components_count(mask: np.ndarray) -> int:
    n, labels, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    return max(0, n - 1)


def quality_from_crop_and_mask(crop: np.ndarray, mask: np.ndarray, bbox: list[float], image_shape: tuple[int, int]) -> dict[str, float]:
    h, w = image_shape
    x1, y1, x2, y2 = bbox
    bbox_area = max(1.0, (x2 - x1) * (y2 - y1))
    area = float((mask > 0).sum())
    fill = area / bbox_area
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop is not None and crop.size else np.zeros((1, 1), dtype=np.uint8)
    blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    brightness = float(gray.mean() / 255.0)
    contrast = float(gray.std() / 64.0)
    fragments = connected_components_count(mask)
    blur_score = min(1.0, blur / 120.0)
    brightness_score = max(0.0, 1.0 - abs(brightness - 0.5) * 1.6)
    contrast_score = min(1.0, contrast)
    fill_score = float(np.clip((fill - 0.05) / 0.55, 0, 1))
    frag_score = 1.0 if fragments <= 1 else max(0.2, 1.0 / fragments)
    mask_quality = 0.35 * fill_score + 0.20 * blur_score + 0.15 * brightness_score + 0.15 * contrast_score + 0.15 * frag_score
    return {"mask_area": area, "bbox_area": bbox_area, "mask_bbox_fill_ratio": fill, "blur_score": blur, "brightness_score": brightness_score, "contrast_score": contrast_score, "edge_cutoff_ratio": 0.0, "mask_fragment_count": fragments, "mask_quality_score": float(np.clip(mask_quality, 0, 1))}


def enrich_detection(frame_rec: dict[str, Any], det: dict[str, Any], depth: np.ndarray, embedder: AppearanceEmbedder) -> dict[str, Any] | None:
    frame_bgr = cv2.imread(frame_rec["frame_path"])
    if frame_bgr is None:
        return None
    h, w = frame_bgr.shape[:2]
    mask = cv2.imread(det["mask_path"], cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    m = mask > 0
    if int(m.sum()) < CONFIG["MIN_MASK_AREA"]:
        return None
    depth_pixels = depth[m]
    if len(depth_pixels) < CONFIG["MIN_DEPTH_PIXELS"]:
        return None
    x1, y1, x2, y2 = [float(v) for v in det["bbox"]]
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    crop = cv2.imread(det["crop_path"]) if det.get("crop_path") else frame_bgr[int(y1):int(y2), int(x1):int(x2)]
    q = quality_from_crop_and_mask(crop, mask, [x1, y1, x2, y2], (h, w))
    p20, p80 = np.percentile(depth_pixels, [20, 80])
    med = float(np.median(depth_pixels))
    masked_crop_rgb, mask_src = make_masked_crop(frame_bgr, [x1, y1, x2, y2], mask, CONFIG["MASK_BACKGROUND_MODE"])
    masked_crop_path = DIRS["debug"] / "masked_embedding_crops" / f"masked_{int(frame_rec['extracted_frame_idx']):06d}_{det['det_id']}.jpg"
    masked_crop_path.parent.mkdir(parents=True, exist_ok=True)
    if masked_crop_rgb is not None and not masked_crop_path.exists():
        cv2.imwrite(str(masked_crop_path), cv2.cvtColor(masked_crop_rgb, cv2.COLOR_RGB2BGR))
    emb, emb_cache = embedder.embed(masked_crop_rgb, det.get("crop_path", ""), det.get("mask_path", ""), mask_src) if masked_crop_rgb is not None else (None, None)
    ar = (x2 - x1) / max(1.0, y2 - y1)
    enriched = dict(det)
    enriched.update(q)
    enriched.update({
        "extracted_frame_idx": frame_rec["extracted_frame_idx"], "original_frame_idx": frame_rec["original_frame_idx"], "frame_path": frame_rec["frame_path"],
        "center_2d": [cx, cy], "center_2d_norm": [cx / w, cy / h], "median_depth": med, "depth_p20": float(p20), "depth_p80": float(p80),
        "depth_std": float(np.std(depth_pixels)), "depth_range": float(p80 - p20), "pseudo_3d_center": [cx / w, cy / h, med],
        "pseudo_3d_extent": [(x2 - x1) / w, (y2 - y1) / h, float(p80 - p20)], "aspect_ratio": float(ar),
        "masked_crop_path": str(masked_crop_path), "embedding": emb, "embedding_model": embedder.model_name, "embedding_cache_path": emb_cache,
        "embedding_mask_source": mask_src, "appearance_quality_score": 1.0 if emb else 0.0,
        "appearance": emb if emb is not None else color_hist_rgb(masked_crop_rgb) if masked_crop_rgb is not None else [],
        "quality_score": float(0.45 * q["mask_quality_score"] + 0.35 * float(det.get("conf", 0)) + 0.20 * max(0, 1 - min(1, float(np.std(depth_pixels)) * 4))),
    })
    return enriched


enriched_cache = DIRS["debug"] / f"enriched_detections_antimerge_{CONFIG['APPEARANCE_MODEL']}_{CONFIG['MASK_BACKGROUND_MODE']}.json"
if CONFIG.get("REUSE_ENRICHED_DETECTIONS", True) and enriched_cache.exists():
    print(f"Reusing enriched detections: {enriched_cache}")
    enriched_by_frame = json.loads(enriched_cache.read_text(encoding="utf-8"))
else:
    depth_wrapper = DepthWrapper(CONFIG["DEPTH_MODEL"], CONFIG["DEVICE"], CONFIG["DEPTH_INVERT"])
    embedder = AppearanceEmbedder(CONFIG["APPEARANCE_MODEL"], CONFIG["DEVICE"])
    enriched_by_frame = []
    depth = None
    for frame_rec in tqdm(proposals_by_frame, desc="depth + embedding + enrich"):
        idx = int(frame_rec["extracted_frame_idx"])
        if idx % max(1, int(CONFIG["DEPTH_EVERY_N_EXTRACTED_FRAMES"])) == 0 or depth is None:
            depth = load_or_compute_depth(frame_rec, depth_wrapper)
        detections = []
        for det in frame_rec["detections"]:
            e = enrich_detection(frame_rec, det, depth, embedder)
            if e is not None:
                detections.append(e)
        enriched_by_frame.append({**frame_rec, "detections": detections})
    enriched_cache.write_text(json.dumps(enriched_by_frame, indent=2, default=str), encoding="utf-8")
print("enriched detections:", sum(len(fr["detections"]) for fr in enriched_by_frame))

sample = next((fr for fr in enriched_by_frame if fr["detections"]), None)
if sample:
    rgb = cv2.cvtColor(cv2.imread(sample["frame_path"]), cv2.COLOR_BGR2RGB)
    depth_png = next(DIRS["depth"].glob(f"depth_*_{int(sample['extracted_frame_idx']):06d}.png"), None)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(overlay_frame(sample)); axes[0].set_title("RGB + masks"); axes[0].axis("off")
    if depth_png:
        axes[1].imshow(cv2.cvtColor(cv2.imread(str(depth_png)), cv2.COLOR_BGR2RGB)); axes[1].set_title("depth"); axes[1].axis("off")
    plt.tight_layout(); plt.show()


## 5. Implement ByteTrack-style Two-Stage 2.5D Tracker

In [ ]:

COARSE_LABEL_MAP = {
    "pot": "cookware", "pan": "cookware", "wok": "cookware", "cookware": "cookware", "lid": "lid_or_cookware",
    "bowl": "dishware", "plate": "dishware", "dishware": "dishware",
    "cup": "container", "bottle": "container", "jar": "container", "container": "container",
    "box": "container", "package": "container", "bag": "container", "carton": "container", "can": "container",
    "knife": "cutting_tool", "scissors": "cutting_tool", "cutting board": "cutting_tool",
    "spoon": "utensil", "fork": "utensil", "spatula": "utensil", "tongs": "utensil", "ladle": "utensil", "whisk": "utensil", "peeler": "utensil", "utensil": "utensil",
    "hand": "hand", "glove": "hand", "person": "person",
}


def norm_label(label: str) -> str:
    return str(label or "unknown").strip().lower().replace("_", " ")


def coarse_label(label: str) -> str:
    l = norm_label(label)
    return COARSE_LABEL_MAP.get(l, l)


def embedding_or_none(*values):
    for value in values:
        if value is None:
            continue
        arr = np.asarray(value, dtype=np.float32).reshape(-1)
        if arr.size and np.all(np.isfinite(arr)):
            return arr.tolist()
    return None


def has_embedding(value) -> bool:
    return embedding_or_none(value) is not None



def bbox_iou(a, b) -> float:
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1, ix2, iy2 = max(ax1, bx1), max(ay1, by1), min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(1e-6, (ax2 - ax1) * (ay2 - ay1)); area_b = max(1e-6, (bx2 - bx1) * (by2 - by1))
    return float(inter / (area_a + area_b - inter + 1e-6))


def bbox_area_xyxy(b):
    return max(1e-6, (float(b[2])-float(b[0])) * (float(b[3])-float(b[1])))


def center_score(track, det):
    a = np.asarray(track["center_2d"], dtype=np.float32); b = np.asarray(det["center_2d"], dtype=np.float32)
    bw = max(1, det["bbox"][2] - det["bbox"][0]); bh = max(1, det["bbox"][3] - det["bbox"][1])
    d = float(np.linalg.norm(a - b) / max(bw, bh, 1))
    return math.exp(-d)


def depth_score(track, det):
    d = abs(float(track["depth_median"]) - float(det["median_depth"]))
    return math.exp(-d / 0.12)


def mask_score(track, det):
    fill_a = float(track.get("mask_bbox_fill_ratio", 0.5)); fill_b = float(det.get("mask_bbox_fill_ratio", 0.5))
    return math.exp(-abs(fill_a - fill_b) / 0.35)


def size_shape_score(track, det):
    area_ratio = max(bbox_area_xyxy(track["bbox"]), bbox_area_xyxy(det["bbox"])) / max(1e-6, min(bbox_area_xyxy(track["bbox"]), bbox_area_xyxy(det["bbox"])))
    ar_a = float(track.get("aspect_ratio", (track["bbox"][2]-track["bbox"][0]) / max(1, track["bbox"][3]-track["bbox"][1])))
    ar_b = float(det.get("aspect_ratio", (det["bbox"][2]-det["bbox"][0]) / max(1, det["bbox"][3]-det["bbox"][1])))
    ar_score = math.exp(-abs(math.log(max(1e-6, ar_a / max(1e-6, ar_b)))) / 0.8)
    area_score = math.exp(-math.log(area_ratio) / 1.0)
    ext_a = np.asarray(track.get("pseudo_3d_extent", [0,0,0]), dtype=np.float32)
    ext_b = np.asarray(det.get("pseudo_3d_extent", [0,0,0]), dtype=np.float32)
    ext_score = math.exp(-float(np.linalg.norm(ext_a - ext_b)) / 0.35)
    return float(np.clip(0.45 * area_score + 0.35 * ar_score + 0.20 * ext_score, 0, 1))


def motion_score(track, det):
    prev = track.get("prev_center_2d")
    if prev is None:
        return 0.7
    pred = np.asarray(track["center_2d"], dtype=np.float32) + (np.asarray(track["center_2d"], dtype=np.float32) - np.asarray(prev, dtype=np.float32))
    d = np.linalg.norm(pred - np.asarray(det["center_2d"], dtype=np.float32)) / max(1, det["bbox"][2]-det["bbox"][0], det["bbox"][3]-det["bbox"][1])
    return float(math.exp(-d))


def label_score(track, det):
    a = coarse_label(track.get("display_label") or track.get("label", "")); b = coarse_label(det.get("raw_label", ""))
    if a == b: return 1.0
    related = {("cookware", "lid_or_cookware"), ("utensil", "cutting_tool")}
    if (a,b) in related or (b,a) in related: return 0.5
    return 0.2


def gallery_embeddings(track):
    gal = track.get("appearance_gallery", {})
    out = []
    proto = embedding_or_none(gal.get("prototype"))
    if proto is not None:
        out.append(proto)
    out += [embedding_or_none(r.get("embedding")) for r in gal.get("best", []) if embedding_or_none(r.get("embedding")) is not None]
    out += [embedding_or_none(r.get("embedding")) for r in gal.get("recent", []) if embedding_or_none(r.get("embedding")) is not None]
    return out


def get_track_appearance_score(track, det):
    emb = embedding_or_none(det.get("embedding"), det.get("appearance"))
    if emb is None:
        return 0.5
    embs = gallery_embeddings(track)
    if not embs:
        base = embedding_or_none(track.get("appearance"))
        return cosine_similarity(base, emb, default=0.5) if base is not None else 0.5
    return float(np.clip(max(cosine_similarity(e, emb, default=0.5) for e in embs), 0, 1))


def update_track_gallery(track, det, in_conflict=False):
    emb = embedding_or_none(det.get("embedding"), det.get("appearance"))
    if emb is None or det.get("quality_score", 0) < CONFIG["GALLERY_MIN_QUALITY_TO_ADD"]:
        return False
    if CONFIG["DO_NOT_UPDATE_GALLERY_IN_CONFLICT"] and in_conflict:
        return False
    gal = track.setdefault("appearance_gallery", {"best": [], "recent": [], "prototype": None})
    existing = gallery_embeddings(track)
    if existing and max(cosine_similarity(e, emb, default=0) for e in existing) < CONFIG["GALLERY_MIN_APPEARANCE_TO_EXISTING"]:
        return False
    rec = {"embedding": emb, "frame_idx": det["extracted_frame_idx"], "quality_score": det["quality_score"], "crop_path": det.get("crop_path", ""), "mask_path": det.get("mask_path", "")}
    gal["recent"].append(rec); gal["recent"] = gal["recent"][-CONFIG["GALLERY_MAX_RECENT"]:]
    gal["best"].append(rec); gal["best"] = sorted(gal["best"], key=lambda r: r.get("quality_score", 0), reverse=True)[:CONFIG["GALLERY_MAX_BEST"]]
    arrs = [np.asarray(r["embedding"], dtype=np.float32) for r in gal["best"] if embedding_or_none(r.get("embedding")) is not None]
    if arrs:
        proto = np.mean(arrs, axis=0); proto = proto / max(1e-8, np.linalg.norm(proto)); gal["prototype"] = proto.tolist()
    return True


def center_dist_norm(a, b, shape=(480,640)):
    aa = np.asarray(a, dtype=np.float32); bb = np.asarray(b, dtype=np.float32)
    return float(np.linalg.norm((aa-b) / np.asarray([shape[1], shape[0]], dtype=np.float32)))


def detect_conflict_zones(detections, tracks=None, config=None):
    config = config or CONFIG
    for d in detections:
        d["in_conflict_zone"] = False; d["conflict_with_det_ids"] = []; d["conflict_reason"] = ""
    if not config.get("ENABLE_CONFLICT_ZONE", True):
        return detections
    for i, a in enumerate(detections):
        for j, b in enumerate(detections[i+1:], i+1):
            iou = bbox_iou(a["bbox"], b["bbox"])
            cd = float(np.linalg.norm(np.asarray(a["center_2d_norm"]) - np.asarray(b["center_2d_norm"])))
            dd = abs(float(a["median_depth"]) - float(b["median_depth"]))
            reason = None
            if iou > config["CONFLICT_PAIRWISE_IOU_THRESHOLD"]:
                reason = "overlap"
            elif cd < config["CONFLICT_CENTER_DISTANCE_THRESHOLD"] and dd < config["CONFLICT_DEPTH_DIFF_THRESHOLD"]:
                reason = "nearby_same_depth"
            if reason:
                a["in_conflict_zone"] = b["in_conflict_zone"] = True
                a["conflict_with_det_ids"].append(b.get("det_id")); b["conflict_with_det_ids"].append(a.get("det_id"))
                a["conflict_reason"] = b["conflict_reason"] = reason
    return detections


def passes_hard_gates(track, det, comps):
    reasons = []
    if not CONFIG.get("ENABLE_HARD_GATES", True):
        return True, reasons
    has_gallery = bool(gallery_embeddings(track))
    app = comps.get("appearance", 0.5)
    in_conflict = bool(det.get("in_conflict_zone") or track.get("current_conflict_zone"))
    if CONFIG.get("APPEARANCE_MODEL") != "none" and has_gallery:
        min_app = CONFIG["MIN_APPEARANCE_IN_CONFLICT"] if in_conflict else CONFIG["MIN_APPEARANCE_FOR_MATCH"]
        if app < min_app:
            reasons.append("appearance_too_low_in_conflict" if in_conflict else "appearance_too_low")
    size_ratio = max(bbox_area_xyxy(track["bbox"]), bbox_area_xyxy(det["bbox"])) / max(1e-6, min(bbox_area_xyxy(track["bbox"]), bbox_area_xyxy(det["bbox"])))
    if size_ratio > CONFIG["MAX_SIZE_CHANGE_RATIO"] and app < 0.75:
        reasons.append("size_change_too_large")
    if comps.get("iou", 0) < CONFIG["MIN_IOU_TO_IGNORE_DEPTH_GATE"] and abs(float(det["median_depth"]) - float(track["depth_median"])) > CONFIG["MAX_DEPTH_DIFF_FOR_LOW_IOU"]:
        reasons.append("depth_diff_too_large_for_low_iou")
    return len(reasons) == 0, reasons


def association_score(track, det, weights=None):
    w = weights or {"iou": CONFIG["W_IOU"], "center": CONFIG["W_CENTER"], "depth": CONFIG["W_DEPTH"], "mask": CONFIG["W_MASK"], "size_shape": CONFIG["W_SIZE_SHAPE"], "appearance": CONFIG["W_APPEARANCE"], "label": CONFIG["W_LABEL"], "motion": CONFIG["W_MOTION"]}
    comps = {"iou": bbox_iou(track["bbox"], det["bbox"]), "center": center_score(track, det), "depth": depth_score(track, det), "mask": mask_score(track, det), "size_shape": size_shape_score(track, det), "appearance": get_track_appearance_score(track, det), "label": label_score(track, det), "motion": motion_score(track, det)}
    ok, reasons = passes_hard_gates(track, det, comps)
    score = sum(w[k] * comps[k] for k in w)
    return float(score), comps, ok, reasons


def is_occluded(track, current_dets):
    for det in current_dets:
        if bbox_iou(track["bbox"], det["bbox"]) > 0.15 and float(det["median_depth"]) < float(track["depth_median"]) - 0.03:
            return True
    if any(d.get("in_conflict_zone") and bbox_iou(track["bbox"], d["bbox"]) > 0.05 for d in current_dets):
        return True
    return False


class TwoStage25DTracker:
    def __init__(self):
        self.next_id = 1; self.active = []; self.finished = []; self.history = defaultdict(list); self.match_logs = []; self.rejected_matches = []

    def new_track(self, det, frame_idx: int):
        t = {"track_id": self.next_id, "video_id": Path(CONFIG["VIDEO_PATH"]).stem, "start_frame": frame_idx, "last_seen_frame": frame_idx, "bbox": det["bbox"], "mask_path": det["mask_path"], "crop_path": det["crop_path"], "center_2d": det["center_2d"], "prev_center_2d": None, "center_3d": det["pseudo_3d_center"], "depth_median": det["median_depth"], "depth_range": [det["depth_p20"], det["depth_p80"]], "pseudo_3d_extent": det.get("pseudo_3d_extent"), "aspect_ratio": det.get("aspect_ratio"), "appearance": embedding_or_none(det.get("embedding"), det.get("appearance")) or [], "appearance_gallery": {"best": [], "recent": [], "prototype": None}, "mask_bbox_fill_ratio": det.get("mask_bbox_fill_ratio", 0.5), "label": det["raw_label"], "display_label": det["raw_label"], "label_history": [det["raw_label"]], "raw_label_history": [det["raw_label"]], "conf_history": [det["conf"]], "quality_history": [det["quality_score"]], "depth_history": [det["median_depth"]], "status_history": ["visible"], "conflict_history": [bool(det.get("in_conflict_zone"))], "match_appearance_history": [1.0], "match_score_history": [1.0], "anti_merge_reject_count": 0, "age": 1, "missed": 0, "status": "visible", "candidate_frames": []}
        det = dict(det); det.update({"appearance_score_to_track": 1.0, "match_score": 1.0, "match_component_scores": {}, "gallery_updated": False, "export_eligible": not det.get("in_conflict_zone", False)})
        t["candidate_frames"].append(det)
        update_track_gallery(t, det, in_conflict=bool(det.get("in_conflict_zone")))
        self.next_id += 1; self.active.append(t)

    def update_track(self, t, det, frame_idx: int, match_score, comps, low_conf_extension=False):
        in_conflict = bool(det.get("in_conflict_zone") or t.get("current_conflict_zone"))
        a = CONFIG["CONFLICT_WEAK_UPDATE_ALPHA"] if in_conflict else CONFIG["EMA_ALPHA"]
        old_center = list(t["center_2d"])
        old3 = np.asarray(t["center_3d"], dtype=np.float32); new3 = np.asarray(det["pseudo_3d_center"], dtype=np.float32)
        t["center_3d"] = (a * old3 + (1 - a) * new3).tolist(); t["depth_median"] = float(a * float(t["depth_median"]) + (1 - a) * float(det["median_depth"]))
        t["prev_center_2d"] = old_center; t["bbox"] = det["bbox"]; t["mask_path"] = det["mask_path"]; t["crop_path"] = det["crop_path"]; t["center_2d"] = det["center_2d"]; t["depth_range"] = [det["depth_p20"], det["depth_p80"]]; t["pseudo_3d_extent"] = det.get("pseudo_3d_extent"); t["aspect_ratio"] = det.get("aspect_ratio"); t["mask_bbox_fill_ratio"] = det.get("mask_bbox_fill_ratio", t.get("mask_bbox_fill_ratio", 0.5)); t["last_seen_frame"] = frame_idx; t["age"] += 1; t["missed"] = 0; t["status"] = "visible"; t["current_conflict_zone"] = in_conflict
        if not in_conflict:
            t["label_history"].append(det["raw_label"]); t["raw_label_history"].append(det["raw_label"]); t["conf_history"].append(det["conf"]); t["quality_history"].append(det["quality_score"] * (0.75 if low_conf_extension else 1.0))
        t["depth_history"].append(det["median_depth"]); t["status_history"].append("conflict_update" if in_conflict else "visible"); t["conflict_history"].append(in_conflict); t["match_appearance_history"].append(comps.get("appearance", 0.5)); t["match_score_history"].append(match_score)
        det = dict(det); gallery_updated = update_track_gallery(t, det, in_conflict=in_conflict)
        det.update({"appearance_score_to_track": comps.get("appearance", 0.5), "size_shape_score_to_track": comps.get("size_shape", 0.5), "match_score": match_score, "match_component_scores": comps, "gallery_updated": gallery_updated, "export_eligible": (not in_conflict and comps.get("appearance", 0.5) >= CONFIG["MIN_APPEARANCE_FOR_MATCH"] and gallery_updated)})
        if len(t["candidate_frames"]) < CONFIG["MAX_CANDIDATES_PER_TRACK"] and det["quality_score"] >= 0.35:
            t["candidate_frames"].append(det)

    def match(self, tracks, detections, threshold, weights=None, frame_idx=0):
        pairs = []
        for ti, t in enumerate(tracks):
            t["current_conflict_zone"] = any(d.get("in_conflict_zone") and bbox_iou(t["bbox"], d["bbox"]) > 0.05 for d in detections)
            for di, d in enumerate(detections):
                score, comps, ok, reasons = association_score(t, d, weights=weights)
                if not ok:
                    self.rejected_matches.append({"frame_idx": frame_idx, "track_id": t["track_id"], "det_id": d.get("det_id"), "score": score, "reasons": reasons, "components": comps})
                    t["anti_merge_reject_count"] = t.get("anti_merge_reject_count", 0) + 1
                pairs.append((score, ti, di, comps, ok, reasons))
        use_hungarian = CONFIG["MATCHING_STRATEGY"] == "hungarian" and (not CONFIG["USE_HUNGARIAN_IN_CONFLICT_ONLY"] or any(d.get("in_conflict_zone") for d in detections))
        matches=[]; mt=set(); md=set()
        if use_hungarian and pairs:
            try:
                from scipy.optimize import linear_sum_assignment
                mat = np.full((len(tracks), len(detections)), 1e6, dtype=np.float32); info={}
                for score,ti,di,comps,ok,reasons in pairs:
                    if ok: mat[ti,di] = 1-score
                    info[(ti,di)] = (score, comps, ok, reasons)
                rows, cols = linear_sum_assignment(mat)
                for ti,di in zip(rows,cols):
                    score, comps, ok, reasons = info[(ti,di)]
                    if ok and score >= threshold:
                        matches.append((ti,di,score,comps)); mt.add(ti); md.add(di)
            except Exception as exc:
                print("Hungarian unavailable, falling back to greedy:", exc); use_hungarian=False
        if not use_hungarian:
            pairs.sort(reverse=True, key=lambda x:x[0])
            for score,ti,di,comps,ok,reasons in pairs:
                if score < threshold or not ok or ti in mt or di in md: continue
                matches.append((ti,di,score,comps)); mt.add(ti); md.add(di)
        for ti,di,score,comps in matches:
            self.match_logs.append({"frame_idx": frame_idx, "track_id": tracks[ti]["track_id"], "det_id": detections[di].get("det_id"), "final_score": score, "component_scores": comps, "in_conflict_zone": bool(detections[di].get("in_conflict_zone"))})
        return matches, mt, md

    def step(self, frame_idx:int, detections:list[dict[str,Any]]):
        detections = detect_conflict_zones(detections, self.active, CONFIG)
        high=[d for d in detections if d["conf"]>=CONFIG["HIGH_CONF_THRESHOLD"]]; low=[d for d in detections if CONFIG["LOW_CONF_THRESHOLD"]<=d["conf"]<CONFIG["HIGH_CONF_THRESHOLD"]]
        tracks=list(self.active); matches1,mt1,md1=self.match(tracks, high, CONFIG["MATCH_THRESHOLD_STAGE1"], frame_idx=frame_idx)
        unmatched_tracks=[tracks[i] for i in set(range(len(tracks)))-mt1]
        stage2_weights={"iou":CONFIG["W_IOU"],"center":CONFIG["W_CENTER"],"depth":CONFIG["W_DEPTH"]*1.2,"mask":CONFIG["W_MASK"],"size_shape":CONFIG["W_SIZE_SHAPE"],"appearance":CONFIG["W_APPEARANCE"]*1.2,"label":CONFIG["W_LABEL"]*0.5,"motion":CONFIG["W_MOTION"]}
        matches2,mt2,md2=self.match(unmatched_tracks, low, CONFIG["MATCH_THRESHOLD_STAGE2"], weights=stage2_weights, frame_idx=frame_idx)
        matched_ids=set()
        for ti,di,score,comps in matches1:
            self.update_track(tracks[ti], high[di], frame_idx, score, comps, False); matched_ids.add(id(tracks[ti]))
        for lti,di,score,comps in matches2:
            t=unmatched_tracks[lti]; self.update_track(t, low[di], frame_idx, score, comps, True); matched_ids.add(id(t))
        matched_high={di for _,di,_,_ in matches1}; matched_low={di for _,di,_,_ in matches2}
        survivors=[]
        for t in self.active:
            if id(t) in matched_ids: survivors.append(t); continue
            t["missed"]+=1; occ=is_occluded(t,detections); t["status"]="occluded" if occ else "missing"; t["status_history"].append(t["status"]); t.setdefault("conflict_history",[]).append(False)
            limit=CONFIG["MAX_OCCLUDED_FRAMES"] if occ else CONFIG["MAX_MISSING_FRAMES"]
            if t["missed"]<=limit: survivors.append(t)
            else: self.finished.append(t)
        self.active=survivors
        for i,d in enumerate(high):
            if i not in matched_high: self.new_track(d,frame_idx)
        for i,d in enumerate(low):
            if i not in matched_low and d["conf"]>=0.35: self.new_track(d,frame_idx)
        self.history[frame_idx]=[dict(t, appearance=[], appearance_gallery={}) for t in self.active]

    def all_tracks(self):
        by={}
        for t in [*self.finished,*self.active]: by[t["track_id"]]=t
        return list(by.values())


def detect_track_splits(tracks):
    split_rows=[]
    for t in tracks:
        cands=[d for d in t.get("candidate_frames",[]) if d.get("embedding") and d.get("quality_score",0)>=0.45]
        t["split_candidate"] = False; t["split_reason"]=""; t["split_cluster_summary"]={}
        if not CONFIG["ENABLE_TRACK_SPLIT_DETECTION"] or len(cands)<CONFIG["SPLIT_MIN_TRACK_EMBEDDINGS"]: continue
        embs=np.asarray([d["embedding"] for d in cands], dtype=np.float32); proto=embs.mean(axis=0); proto=proto/max(1e-8,np.linalg.norm(proto))
        dists=1-np.clip(embs@proto, -1, 1); intra=float(np.mean(dists))
        if intra <= CONFIG["SPLIT_MAX_INTRA_VARIANCE"]: continue
        try:
            from sklearn.cluster import KMeans
            labels=KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(embs)
        except Exception:
            sim=embs@embs.T; i,j=np.unravel_index(np.argmin(sim), sim.shape); seeds=embs[[i,j]]; labels=np.argmax(embs@seeds.T, axis=1)
        centers=[]; frames=[]
        for k in [0,1]:
            group=embs[labels==k]; center=group.mean(axis=0); center=center/max(1e-8,np.linalg.norm(center)); centers.append(center); frames.append([int(cands[ii]["extracted_frame_idx"]) for ii,l in enumerate(labels) if l==k])
        sep=float(1-np.clip(np.dot(centers[0], centers[1]), -1, 1))
        if sep>CONFIG["SPLIT_MIN_TWO_CLUSTER_SEPARATION"]:
            t["split_candidate"]=True; t["split_reason"]="appearance_two_clusters"; t["split_cluster_summary"]={"cluster_0_frames":frames[0],"cluster_1_frames":frames[1],"cluster_separation":sep,"intra_variance":intra}
            split_rows.append({"track_id":t["track_id"], **t["split_cluster_summary"]})
    (DIRS["tracks"] / "split_candidates.json").write_text(json.dumps(split_rows, indent=2, default=str), encoding="utf-8")
    return tracks

tracks_raw_cache = DIRS["tracks"] / f"tracks_raw_antimerge_{CONFIG['APPEARANCE_MODEL']}_{CONFIG['MATCHING_STRATEGY']}.json"
tracker_25d=None
if CONFIG.get("REUSE_TRACKS_RAW", True) and tracks_raw_cache.exists():
    print(f"Reusing anti-merge raw tracks: {tracks_raw_cache}")
    tracks_raw=json.loads(tracks_raw_cache.read_text(encoding="utf-8"))
else:
    tracker_25d=TwoStage25DTracker()
    for fr in tqdm(enriched_by_frame, desc="2.5D anti-merge tracking"):
        tracker_25d.step(int(fr["extracted_frame_idx"]), fr["detections"])
    tracks_raw=detect_track_splits(tracker_25d.all_tracks())
    tracks_raw_cache.write_text(json.dumps([{k:v for k,v in t.items() if k not in {"appearance"}} for t in tracks_raw], indent=2, default=str), encoding="utf-8")
    with (DIRS["debug"] / "match_logs.jsonl").open("w", encoding="utf-8") as fh:
        for row in tracker_25d.match_logs: fh.write(json.dumps(row, default=str)+"\n")
    (DIRS["debug"] / "rejected_matches.json").write_text(json.dumps(tracker_25d.rejected_matches, indent=2, default=str), encoding="utf-8")
print("anti-merge 2.5D tracks:", len(tracks_raw))


## 6. Track-level Label Smoothing / Coarse Voting

In [ ]:
def entropy_from_counts(counts: dict[str, float]) -> float:
    total = sum(counts.values())
    if total <= 0:
        return 0.0
    probs = [v / total for v in counts.values() if v > 0]
    return float(-sum(p * math.log(p + 1e-9) for p in probs))


def smooth_track_labels(track: dict[str, Any]) -> dict[str, Any]:
    fine_votes = defaultdict(float)
    coarse_votes = defaultdict(float)
    raw_counts = Counter()
    for lab, conf, q, status in zip(track["raw_label_history"], track["conf_history"], track["quality_history"], track["status_history"]):
        vis = 1.0 if status == "visible" else 0.5 if status == "occluded" else 0.2
        weight = float(conf) * float(q) * vis
        fine = norm_label(lab)
        coarse = coarse_label(fine)
        fine_votes[fine] += weight
        coarse_votes[coarse] += weight
        raw_counts[fine] += 1

    fine_total = sum(fine_votes.values()) or 1.0
    coarse_total = sum(coarse_votes.values()) or 1.0
    fine_top = sorted(fine_votes.items(), key=lambda kv: kv[1], reverse=True)
    coarse_top = sorted(coarse_votes.items(), key=lambda kv: kv[1], reverse=True)

    display = fine_top[0][0] if fine_top else track.get("label", "unknown")
    export = coarse_top[0][0] if coarse_top else coarse_label(display)
    coarse_conf = float(coarse_top[0][1] / coarse_total) if coarse_top else 0.0
    fine_conf = float(fine_top[0][1] / fine_total) if fine_top else 0.0
    raw_entropy = entropy_from_counts(dict(fine_votes))
    coarse_entropy = entropy_from_counts(dict(coarse_votes))

    out = dict(track)
    out.update({
        "display_label": display,
        "export_label": export,
        "coarse_label": export,
        # Backward-compatible name, now intentionally coarse-first for export.
        "label_confidence": coarse_conf,
        "coarse_label_confidence": coarse_conf,
        "fine_label_confidence": fine_conf,
        "raw_label_entropy": raw_entropy,
        "coarse_label_entropy": coarse_entropy,
        # Raw/fine entropy is only a serious problem when the coarse vote is also weak.
        "within_coarse_label_jitter": bool(raw_entropy > CONFIG["RAW_LABEL_ENTROPY_REVIEW_THRESHOLD"] and coarse_conf >= 0.75),
        "top_3_label_votes": fine_top[:3],
        "top_3_coarse_votes": coarse_top[:3],
        "raw_label_history_summary": dict(raw_counts),
    })
    return out


tracks_smoothed = [smooth_track_labels(t) for t in tracks_raw]
(DIRS["tracks"] / "tracks_smoothed.json").write_text(json.dumps([{k:v for k,v in t.items() if k != "appearance"} for t in tracks_smoothed], indent=2, default=str), encoding="utf-8")
display(pd.DataFrame([{
    "track_id": t["track_id"], "display_label": t["display_label"], "export_label": t["export_label"],
    "coarse_confidence": t["coarse_label_confidence"], "fine_confidence": t["fine_label_confidence"],
    "raw_entropy": t["raw_label_entropy"], "coarse_entropy": t["coarse_label_entropy"],
    "within_coarse_jitter": t["within_coarse_label_jitter"],
    "length": len(set(d["extracted_frame_idx"] for d in t["candidate_frames"]))
} for t in tracks_smoothed]).sort_values("length", ascending=False).head(10))

## 7. Track-level Quality Scoring

In [ ]:

# Local ndarray-safe vector helpers for this cell. This intentionally overrides any older
# notebook-session cosine() left in the kernel from previous runs.
def embedding_or_none(*values):
    for value in values:
        if value is None:
            continue
        try:
            arr = np.asarray(value, dtype=np.float32).reshape(-1)
        except Exception:
            continue
        if arr.size and np.all(np.isfinite(arr)):
            return arr.tolist()
    return None


def cosine(a, b, default=0.5):
    aa = embedding_or_none(a)
    bb = embedding_or_none(b)
    if aa is None or bb is None:
        return default
    aa = np.asarray(aa, dtype=np.float32)
    bb = np.asarray(bb, dtype=np.float32)
    den = float(np.linalg.norm(aa) * np.linalg.norm(bb))
    if den <= 1e-8:
        return default
    return float(np.clip(np.dot(aa, bb) / den, -1, 1))


def stability_score(values: list[float], scale: float = 0.10) -> float:
    vals = [float(v) for v in values if v is not None and np.isfinite(float(v))]
    if len(vals) < 2:
        return 0.5
    return float(math.exp(-float(np.std(vals)) / scale))


def _candidate_embeddings(track):
    embs = []
    for d in track.get("candidate_frames", []):
        emb = embedding_or_none(d.get("embedding"), d.get("appearance"))
        if emb is not None:
            embs.append(np.asarray(emb, dtype=np.float32))
    return embs


def track_appearance_consistency(track):
    embs = _candidate_embeddings(track)
    if len(embs) < 2:
        gallery = track.get("appearance_gallery", {}) or {}
        return 0.5, 0.5
    proto = np.mean(np.stack(embs), axis=0)
    proto = proto / (np.linalg.norm(proto) + 1e-9)
    sims = [cosine(proto, e) for e in embs]
    mean_sim = float(np.mean(sims)) if sims else 0.5
    return mean_sim, float(1.0 - mean_sim)


def size_shape_stability(track):
    cands = track.get("candidate_frames", [])
    if len(cands) < 2:
        return 0.5
    areas, aspects, fills = [], [], []
    for d in cands:
        x1, y1, x2, y2 = d.get("bbox", [0, 0, 0, 0])
        w, h = max(1.0, float(x2) - float(x1)), max(1.0, float(y2) - float(y1))
        areas.append(w * h)
        aspects.append(w / h)
        fills.append(float(d.get("mask_bbox_fill_ratio", 0.0)))
    def cv_score(vals):
        vals = np.asarray(vals, dtype=np.float32)
        if len(vals) < 2 or float(np.mean(vals)) <= 1e-6:
            return 0.5
        cv = float(np.std(vals) / (np.mean(vals) + 1e-6))
        return float(np.exp(-cv))
    return float(np.mean([cv_score(areas), cv_score(aspects), cv_score(fills)]))


def frame_quality_for_export(d):
    return float(np.clip(
        0.35 * float(d.get("quality_score", 0.0))
        + 0.30 * float(d.get("mask_quality_score", 0.0))
        + 0.20 * float(d.get("appearance_quality_score", 0.5))
        + 0.15 * float(d.get("conf", 0.0)),
        0,
        1,
    ))


rejected_match_rows = []
rejected_path = DIRS["debug"] / "rejected_matches.json"
if rejected_path.exists():
    try:
        rejected_match_rows = json.loads(rejected_path.read_text(encoding="utf-8"))
    except Exception:
        rejected_match_rows = []
anti_merge_rejects_by_track = Counter(int(r.get("track_id", -1)) for r in rejected_match_rows if r.get("track_id") is not None)
rejected_reasons_counter = Counter(reason for r in rejected_match_rows for reason in r.get("reasons", []))


def score_track(track: dict[str, Any]) -> dict[str, Any]:
    cands = track.get("candidate_frames", [])
    frames = [d.get("extracted_frame_idx") for d in cands]
    length = len(set(frames))
    status_history = track.get("status_history", [])
    visible_count = sum(1 for s in status_history if s in {"visible", "conflict_update"})
    occluded_count = sum(1 for s in status_history if s == "occluded")
    missing_count = sum(1 for s in status_history if s in {"missing", "ambiguous_missing"})
    status_total = max(1, len(status_history))
    avg_conf = float(np.mean(track.get("conf_history", [0]))) if track.get("conf_history") else 0.0
    avg_mask_q = float(np.mean([d.get("mask_quality_score", 0) for d in cands])) if cands else 0.0
    avg_crop_q = float(np.mean([d.get("quality_score", 0) for d in cands])) if cands else 0.0
    depth_stab = stability_score(track.get("depth_history", []), scale=0.10)
    app_cons, app_var = track_appearance_consistency(track)
    size_stab = size_shape_stability(track)
    conflict_count = sum(1 for d in cands if d.get("in_conflict_zone") or d.get("status") == "conflict_update")
    conflict_ratio = conflict_count / max(1, len(cands))
    gallery = track.get("appearance_gallery", {}) or {}
    gallery_size = len(gallery.get("best", []) or []) + len(gallery.get("recent", []) or [])
    avg_match_app = float(np.mean([d.get("appearance_score_to_track", app_cons) for d in cands])) if cands else app_cons
    anti_merge_reject_count = int(anti_merge_rejects_by_track.get(int(track.get("track_id", -1)), 0))
    coarse_conf = float(track.get("coarse_label_confidence", track.get("label_confidence", 0.0)) or 0.0)
    fine_conf = float(track.get("fine_label_confidence", track.get("label_confidence", 0.0)) or 0.0)
    length_score = min(1.0, length / 12)
    visibility = visible_count / status_total
    false_risk = 1.0 - min(1.0, 0.5 * avg_conf + 0.5 * avg_mask_q)
    split_candidate = bool(track.get("split_candidate", False))

    q = (
        0.12 * length_score
        + 0.12 * visibility
        + 0.12 * avg_mask_q
        + 0.12 * depth_stab
        + 0.18 * app_cons
        + 0.12 * coarse_conf
        + 0.10 * size_stab
        + 0.07 * (1.0 - conflict_ratio)
        + 0.05 * (1.0 - false_risk)
    )

    review_flags, warning_flags = [], []
    if coarse_conf < CONFIG["LABEL_CONFIDENCE_REVIEW_THRESHOLD"]:
        review_flags.append("low_coarse_label_confidence")
    if track.get("raw_label_entropy", 0.0) > CONFIG["RAW_LABEL_ENTROPY_REVIEW_THRESHOLD"]:
        if coarse_conf >= 0.75:
            warning_flags.append("fine_label_jitter_within_coarse")
        else:
            review_flags.append("high_raw_label_entropy")
    if app_cons < CONFIG["MIN_TRACK_APPEARANCE_CONSISTENCY_FOR_EXPORT"]:
        review_flags.append("low_appearance_consistency")
    if conflict_ratio > 0.35:
        review_flags.append("high_conflict_frame_ratio")
    elif conflict_ratio > 0.10:
        warning_flags.append("some_conflict_frames")
    if split_candidate:
        review_flags.append("split_candidate")
    if anti_merge_reject_count >= 3:
        review_flags.append("repeated_anti_merge_rejections")
    if gallery_size < 2 and CONFIG.get("APPEARANCE_MODEL") != "none":
        warning_flags.append("gallery_too_small")
    if size_stab < 0.55:
        review_flags.append("size_shape_instability")
    if depth_stab < CONFIG["DEPTH_STABILITY_REVIEW_THRESHOLD"]:
        warning_flags.append("low_depth_stability")
    if avg_mask_q < CONFIG["MASK_QUALITY_REVIEW_THRESHOLD"]:
        if avg_mask_q < 0.45:
            review_flags.append("low_mask_quality")
        else:
            warning_flags.append("borderline_mask_quality")
    if length < 2:
        review_flags.append("too_short")

    severe = {"too_short", "low_mask_quality", "low_coarse_label_confidence", "split_candidate", "low_appearance_consistency", "size_shape_instability"}
    has_severe = any(f in severe for f in review_flags)
    export_clean = (
        app_cons >= CONFIG["MIN_TRACK_APPEARANCE_CONSISTENCY_FOR_EXPORT"]
        and (not split_candidate or not CONFIG.get("REJECT_SPLIT_CANDIDATES_FROM_GOLD_SILVER", True))
        and conflict_ratio <= 0.25
        and size_stab >= 0.55
    )

    if (
        q >= CONFIG["MIN_TRACK_QUALITY_FOR_GOLD_CANDIDATE"]
        and length >= CONFIG["MIN_TRACK_LENGTH_FOR_SILVER"]
        and coarse_conf >= 0.80
        and avg_mask_q >= 0.60
        and export_clean
        and not review_flags
    ):
        category = "gold"
    elif (
        q >= 0.62
        and length >= CONFIG["MIN_TRACK_LENGTH_FOR_SILVER"]
        and coarse_conf >= 0.60
        and avg_mask_q >= 0.45
        and export_clean
        and not has_severe
    ):
        category = "silver"
    elif length < 2 or avg_mask_q < 0.25 or false_risk > 0.80:
        category = "reject"
    else:
        category = "review"

    if category in {"gold", "silver"}:
        needs_review = False
    else:
        needs_review = category == "review"

    out = dict(track)
    out.update({
        "track_length": length,
        "visible_count": visible_count,
        "occluded_count": occluded_count,
        "missing_count": missing_count,
        "visibility_ratio": visibility,
        "average_conf": avg_conf,
        "average_mask_quality": avg_mask_q,
        "average_crop_quality": avg_crop_q,
        "average_depth_stability": depth_stab,
        "average_appearance_consistency": app_cons,
        "appearance_consistency": app_cons,
        "appearance_variance": app_var,
        "conflict_frame_ratio": conflict_ratio,
        "gallery_size": gallery_size,
        "anti_merge_reject_count": anti_merge_reject_count,
        "average_match_appearance_score": avg_match_app,
        "size_shape_stability_score": size_stab,
        "length_score": length_score,
        "visibility_score": visibility,
        "mask_quality_component": avg_mask_q,
        "depth_stability_score": depth_stab,
        "appearance_consistency_score": app_cons,
        "label_confidence_score": coarse_conf,
        "fine_label_confidence_score": fine_conf,
        "fragmentation_risk_score": max(0.0, 1.0 - length_score),
        "false_object_risk_score": false_risk,
        "track_quality_score": float(np.clip(q, 0, 1)),
        "needs_review": needs_review,
        "review_flags": sorted(set(review_flags)),
        "warning_flags": sorted(set(warning_flags)),
        "category": category,
    })
    return out


tracks_scored = [score_track(t) for t in tracks_smoothed]
(DIRS["tracks"] / "tracks_scored.json").write_text(json.dumps(tracks_scored, indent=2, default=str), encoding="utf-8")
summary_rows = [{
    "track_id": t["track_id"],
    "export_label": t.get("export_label"),
    "quality": t["track_quality_score"],
    "coarse_label_confidence": t.get("coarse_label_confidence", t.get("label_confidence", 0)),
    "fine_label_confidence": t.get("fine_label_confidence", 0),
    "appearance_consistency": t.get("appearance_consistency", 0),
    "conflict_frame_ratio": t.get("conflict_frame_ratio", 0),
    "split_candidate": t.get("split_candidate", False),
    "length": t["track_length"],
    "raw_label_entropy": t.get("raw_label_entropy", 0),
    "needs_review": t["needs_review"],
    "category": t["category"],
    "review_flags": ";".join(t["review_flags"]),
    "warning_flags": ";".join(t.get("warning_flags", [])),
} for t in tracks_scored]
track_summary_df = pd.DataFrame(summary_rows).sort_values(["category", "quality"], ascending=[True, False])
track_summary_df.to_csv(DIRS["tracks"] / "track_summary.csv", index=False)
display(track_summary_df.head(30))
print("category counts:", Counter(t["category"] for t in tracks_scored))
print("split candidates:", sum(bool(t.get("split_candidate")) for t in tracks_scored))
print("rejected match reasons:", rejected_reasons_counter.most_common(10))


## 8. Representative Crop Selection

In [ ]:

# Local ndarray-safe vector helpers for this cell. This intentionally overrides any older
# notebook-session cosine() left in the kernel from previous runs.
def embedding_or_none(*values):
    for value in values:
        if value is None:
            continue
        try:
            arr = np.asarray(value, dtype=np.float32).reshape(-1)
        except Exception:
            continue
        if arr.size and np.all(np.isfinite(arr)):
            return arr.tolist()
    return None


def cosine(a, b, default=0.5):
    aa = embedding_or_none(a)
    bb = embedding_or_none(b)
    if aa is None or bb is None:
        return default
    aa = np.asarray(aa, dtype=np.float32)
    bb = np.asarray(bb, dtype=np.float32)
    den = float(np.linalg.norm(aa) * np.linalg.norm(bb))
    if den <= 1e-8:
        return default
    return float(np.clip(np.dot(aa, bb) / den, -1, 1))


IMPORTANT_CLASSES = {"hand", "cookware", "lid_or_cookware", "utensil", "cutting_tool", "dishware"}


def label_vote_support(track, raw_label):
    coarse = coarse_label(raw_label)
    votes = dict(track.get("top_3_coarse_votes", []))
    total = sum(votes.values()) or 1.0
    return float(votes.get(coarse, 0.0) / total)


def candidate_depth_stability(track, d):
    depth = float(d.get("median_depth", 0.5))
    hist = [float(v) for v in track.get("depth_history", []) if v is not None]
    if not hist:
        return 0.5
    return float(math.exp(-abs(depth - float(np.median(hist))) / 0.12))


def selection_score(track, d, selected):
    frame_quality = frame_quality_for_export(d)
    app = float(d.get("appearance_score_to_track", track.get("appearance_consistency", 0.5)))
    mask_q = float(d.get("mask_quality_score", 0.0))
    depth_stab = candidate_depth_stability(track, d)
    support = label_vote_support(track, d.get("raw_label", "unknown"))
    diversity = 0.5
    if selected:
        app_dists = [1.0 - cosine(embedding_or_none(d.get("embedding"), d.get("appearance")), embedding_or_none(s.get("embedding"), s.get("appearance")), default=0.5) for s in selected]
        frame_dists = [min(1.0, abs(int(d["extracted_frame_idx"]) - int(s["extracted_frame_idx"])) / 25.0) for s in selected]
        diversity = float(np.clip(0.5 * max(app_dists or [0]) + 0.5 * max(frame_dists or [0]), 0, 1))
    return float(np.clip(
        0.30 * frame_quality
        + 0.25 * app
        + 0.15 * mask_q
        + 0.15 * depth_stab
        + 0.10 * diversity
        + 0.05 * support,
        0,
        1,
    ))


def is_gold_silver_eligible(track, d):
    if track.get("split_candidate") and CONFIG.get("REJECT_SPLIT_CANDIDATES_FROM_GOLD_SILVER", True):
        return False
    if d.get("in_conflict_zone") and CONFIG.get("DO_NOT_EXPORT_CONFLICT_FRAMES", True):
        return False
    if float(d.get("appearance_score_to_track", track.get("appearance_consistency", 0.5))) < CONFIG["MIN_APPEARANCE_FOR_MATCH"]:
        return False
    if not d.get("gallery_updated", False) and CONFIG.get("APPEARANCE_MODEL") != "none":
        return False
    if d.get("match_component_scores", {}).get("size_shape_score", 1.0) < 0.35:
        return False
    if d.get("mask_quality_score", 0.0) < 0.45:
        return False
    return True


def select_representative_crops(track: dict[str, Any]) -> list[dict[str, Any]]:
    base = [
        d for d in track.get("candidate_frames", [])
        if d.get("mask_quality_score", 0) >= 0.30
        and d.get("quality_score", 0) >= 0.30
        and d.get("mask_area", 0) >= CONFIG["MIN_MASK_AREA"]
        and d.get("crop_path")
    ]
    if track["category"] in {"gold", "silver"}:
        cands = [d for d in base if is_gold_silver_eligible(track, d)]
    elif track["category"] == "review":
        cands = base
    else:
        cands = base[: min(len(base), CONFIG["EXPORT_MIN_PER_TRACK"])]

    limit = CONFIG["IMPORTANT_CLASS_EXPORT_MAX"] if track.get("export_label") in IMPORTANT_CLASSES else CONFIG["EXPORT_MAX_PER_TRACK"]
    if track["category"] == "review":
        limit = min(limit, max(CONFIG["EXPORT_MIN_PER_TRACK"], 6))
    if track["category"] == "reject":
        limit = min(limit, 2)

    selected = []
    remaining = list(cands)
    while remaining and len(selected) < limit:
        scored = [(selection_score(track, d, selected), d) for d in remaining]
        scored.sort(key=lambda x: x[0], reverse=True)
        score, d = scored[0]
        selected.append(d)
        new_remaining = []
        for cand in remaining:
            if cand is d:
                continue
            near_frame = abs(int(cand["extracted_frame_idx"]) - int(d["extracted_frame_idx"])) < 3
            too_similar = cosine(embedding_or_none(cand.get("embedding"), cand.get("appearance")), embedding_or_none(d.get("embedding"), d.get("appearance")), default=0.0) > 0.97
            if near_frame and too_similar:
                continue
            new_remaining.append(cand)
        remaining = new_remaining

    if not selected and base:
        selected = sorted(base, key=lambda d: d.get("quality_score", 0), reverse=True)[:CONFIG["EXPORT_MIN_PER_TRACK"]]

    out = []
    for d in selected:
        rec_category = track["category"]
        if track["category"] in {"gold", "silver"} and not is_gold_silver_eligible(track, d):
            rec_category = "review"
        out.append({
            "track_id": track["track_id"],
            "frame_idx": d["extracted_frame_idx"],
            "original_frame_idx": d["original_frame_idx"],
            "crop_path": d["crop_path"],
            "mask_path": d.get("mask_path"),
            "export_label": track["export_label"],
            "raw_label": d.get("raw_label"),
            "quality_score": d.get("quality_score", 0),
            "category": rec_category,
            "track_quality_score": track["track_quality_score"],
            "label_confidence": track.get("coarse_label_confidence", track.get("label_confidence", 0)),
            "fine_label_confidence": track.get("fine_label_confidence", 0),
            "appearance_consistency": track.get("appearance_consistency", 0.5),
            "appearance_score_to_track": d.get("appearance_score_to_track", track.get("appearance_consistency", 0.5)),
            "in_conflict_zone": bool(d.get("in_conflict_zone", False)),
            "split_candidate": bool(track.get("split_candidate", False)),
            "bbox": d.get("bbox"),
            "conf": d.get("conf", 0),
            "depth_median": d.get("median_depth"),
            "mask_quality": d.get("mask_quality_score", 0),
            "review_flags": track.get("review_flags", []),
            "warning_flags": track.get("warning_flags", []),
            "raw_label_history_summary": track.get("raw_label_history_summary", []),
            "metadata": {
                "top_3_label_votes": track.get("top_3_label_votes", []),
                "top_3_coarse_votes": track.get("top_3_coarse_votes", []),
                "selection_score": selection_score(track, d, selected),
                "match_component_scores": d.get("match_component_scores", {}),
                "gallery_updated": d.get("gallery_updated", False),
            },
        })
    return out


selected_crop_records = []
for t in tracks_scored:
    selected_crop_records.extend(select_representative_crops(t))
print("selected crops:", len(selected_crop_records))
if selected_crop_records:
    display(pd.DataFrame(selected_crop_records).head())
print("selected by category:", Counter(r["category"] for r in selected_crop_records))


## 9. Export Dataset Folders

In [ ]:
def safe_label(label: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(label))[:60]


labels_by_split = defaultdict(list)
for rec in selected_crop_records:
    split = rec["category"]
    if split not in {"gold", "silver", "review", "reject"}:
        split = "review"
    label = safe_label(rec["export_label"])
    image_name = f"track_{int(rec['track_id']):04d}_frame_{int(rec['frame_idx']):06d}_{label}.jpg"
    mask_name = f"track_{int(rec['track_id']):04d}_frame_{int(rec['frame_idx']):06d}_{label}_mask.png"
    img_out = OUTPUT_DIR / "export" / split / "images" / image_name
    mask_out = OUTPUT_DIR / "export" / split / "masks" / mask_name
    if rec.get("crop_path") and Path(rec["crop_path"]).exists():
        shutil.copy2(rec["crop_path"], img_out)
    if rec.get("mask_path") and Path(rec["mask_path"]).exists():
        shutil.copy2(rec["mask_path"], mask_out)
    item = dict(rec)
    item.update({
        "image_path": str(img_out),
        "mask_path": str(mask_out),
        "coarse_label": rec["export_label"],
    })
    labels_by_split[split].append(item)

for split, rows in labels_by_split.items():
    out = OUTPUT_DIR / "export" / split / "labels.json"
    out.write_text(json.dumps(rows, indent=2, default=str), encoding="utf-8")

print({k: len(v) for k, v in labels_by_split.items()})

## 10. Visualizations and Review Contact Sheets

In [ ]:
def draw_frame_tracks(frame_rec, tracks_history):
    img = cv2.imread(frame_rec["frame_path"])
    if img is None:
        return None
    out = img.copy()
    for t in tracks_history:
        color = (50 + (t["track_id"] * 37) % 205, 80 + (t["track_id"] * 53) % 175, 120 + (t["track_id"] * 29) % 135)
        x1, y1, x2, y2 = [int(v) for v in t["bbox"]]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2 if t["status"] == "visible" else 1)
        app = t.get("appearance_score_to_track", None)
        conflict = " C" if t.get("in_conflict_zone") else ""
        app_txt = f" app={float(app):.2f}" if app is not None else ""
        text = f"#{t['track_id']} {t.get('display_label', t.get('label'))} z={float(t.get('depth_median',0)):.2f}{app_txt} {t['status']}{conflict}"
        cv2.putText(out, text, (x1, max(16, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return out


vis_video_path = DIRS["visualizations"] / "tracking_2p5d_bytetrack.mp4"
vis_frames = []
if CONFIG.get("REUSE_TRACKING_VISUALIZATION_VIDEO", True) and vis_video_path.exists():
    print(f"Reusing tracking visualization video: {vis_video_path}")
elif tracker_25d is None:
    print("Skipping tracking visualization video because raw tracks were loaded from cache and per-frame tracker history is unavailable.")
else:
    for fr in enriched_by_frame:
        idx = int(fr["extracted_frame_idx"])
        out = draw_frame_tracks(fr, tracker_25d.history.get(idx, []))
        if out is None:
            continue
        depth_png = next(DIRS["depth"].glob(f"depth_*_{idx:06d}.png"), DIRS["depth"] / f"depth_{idx:06d}.png")
        depth_img = cv2.imread(str(depth_png)) if depth_png.exists() else np.zeros_like(out)
        if depth_img is not None and depth_img.shape[:2] != out.shape[:2]:
            depth_img = cv2.resize(depth_img, (out.shape[1], out.shape[0]))
        combo = np.hstack([out, depth_img])
        p = DIRS["visualizations"] / f"tracking_{idx:06d}.jpg"
        cv2.imwrite(str(p), combo)
        vis_frames.append(p)

    if CONFIG["SAVE_VISUALIZATION_VIDEO"] and vis_frames:
        first = cv2.imread(str(vis_frames[0]))
        h, w = first.shape[:2]
        writer = cv2.VideoWriter(str(vis_video_path), cv2.VideoWriter_fourcc(*"mp4v"), 6, (w, h))
        for p in vis_frames:
            im = cv2.imread(str(p))
            if im is not None:
                writer.write(im)
        writer.release()


def make_contact_sheet(track, max_tiles=16):
    cands = track["candidate_frames"][:max_tiles]
    if not cands:
        return
    tile = 160
    cols = 4
    rows = math.ceil(len(cands) / cols)
    canvas = np.full((rows * (tile + 38) + 80, cols * tile, 3), 245, dtype=np.uint8)
    split = " | SPLIT" if track.get("split_candidate") else ""
    title = f"track {track['track_id']} | {track['export_label']} | q={track['track_quality_score']:.2f} | app={track.get('appearance_consistency',0.5):.2f} | conflict={track.get('conflict_frame_ratio',0):.2f} | coarse={track.get('coarse_label_confidence', track.get('label_confidence', 0)):.2f} | {track['category']}{split}"
    cv2.putText(canvas, title[:120], (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (20,20,20), 1)
    cv2.putText(canvas, str(track.get("top_3_label_votes", ""))[:120], (8, 52), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (50,50,50), 1)
    for i, d in enumerate(cands):
        crop = cv2.imread(d["crop_path"])
        if crop is None:
            continue
        crop = cv2.resize(crop, (tile, tile), interpolation=cv2.INTER_AREA)
        r, c = divmod(i, cols)
        y, x = 80 + r * (tile + 38), c * tile
        canvas[y:y+tile, x:x+tile] = crop
        cv2.putText(canvas, f"f{d['extracted_frame_idx']} {d['raw_label']} {d['conf']:.2f}", (x+3, y+tile+16), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (20,20,20), 1)
        cv2.putText(canvas, f"z {d['median_depth']:.2f} q {d['quality_score']:.2f}", (x+3, y+tile+32), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (60,60,60), 1)
        cv2.putText(canvas, f"app {d.get('appearance_score_to_track', track.get('appearance_consistency', 0.5)):.2f} {'C' if d.get('in_conflict_zone') else ''}", (x+3, y+tile+48), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (80,80,80), 1)
    out = DIRS["review_contact_sheets"] / f"track_{int(track['track_id']):04d}_{safe_label(track['export_label'])}.jpg"
    cv2.imwrite(str(out), canvas)


def make_stability_timeline_sheet(track, max_tiles=20):
    cands = sorted(track["candidate_frames"], key=lambda d: d["extracted_frame_idx"])
    if not cands:
        return
    if len(cands) > max_tiles:
        idxs = np.linspace(0, len(cands) - 1, max_tiles).round().astype(int).tolist()
        cands = [cands[i] for i in idxs]
    tile_w, tile_h = 132, 108
    header_h, footer_h = 96, 68
    cols = len(cands)
    canvas = np.full((header_h + tile_h + footer_h, max(1, cols) * tile_w, 3), 248, dtype=np.uint8)
    title = (
        f"track {track['track_id']} | export={track['export_label']} | category={track['category']} | "
        f"q={track['track_quality_score']:.2f} | coarse_conf={track.get('coarse_label_confidence', 0):.2f} | "
        f"fine_conf={track.get('fine_label_confidence', 0):.2f} | len={track['track_length']}"
    )
    cv2.putText(canvas, title[:170], (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (20, 20, 20), 1)
    cv2.putText(canvas, f"coarse votes: {track.get('top_3_coarse_votes', [])}"[:170], (8, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (50, 50, 50), 1)
    cv2.putText(canvas, f"warnings: {track.get('warning_flags', [])} review: {track.get('review_flags', [])}"[:170], (8, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (70, 70, 70), 1)
    depths = [float(d.get("median_depth", 0)) for d in cands]
    dmin, dmax = min(depths), max(depths)
    export_coarse = track["export_label"]
    for i, d in enumerate(cands):
        x = i * tile_w
        y = header_h
        crop = cv2.imread(d["crop_path"])
        if crop is None:
            crop = np.full((tile_h, tile_w, 3), 200, dtype=np.uint8)
        crop = cv2.resize(crop, (tile_w, tile_h), interpolation=cv2.INTER_AREA)
        canvas[y:y+tile_h, x:x+tile_w] = crop
        raw = norm_label(d.get("raw_label", ""))
        raw_coarse = coarse_label(raw)
        ok = raw_coarse == export_coarse
        color = (30, 130, 40) if ok else (30, 80, 220)
        cv2.rectangle(canvas, (x, y), (x + tile_w - 1, y + tile_h - 1), color, 3 if ok else 4)
        z = float(d.get("median_depth", 0))
        q = float(d.get("quality_score", 0))
        bar_h = int((z - dmin) / max(1e-6, dmax - dmin) * 40)
        cv2.rectangle(canvas, (x + tile_w - 12, y + tile_h - 4 - bar_h), (x + tile_w - 4, y + tile_h - 4), (90, 90, 90), -1)
        cv2.putText(canvas, f"f{d['extracted_frame_idx']}", (x + 3, y + tile_h + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (20,20,20), 1)
        cv2.putText(canvas, raw[:14], (x + 3, y + tile_h + 31), cv2.FONT_HERSHEY_SIMPLEX, 0.34, color, 1)
        app = float(d.get("appearance_score_to_track", track.get("appearance_consistency", 0.5)))
        marker = " C" if d.get("in_conflict_zone") else ""
        cv2.putText(canvas, f"z{z:.2f} q{q:.2f} a{app:.2f}{marker}", (x + 3, y + tile_h + 48), cv2.FONT_HERSHEY_SIMPLEX, 0.34, (60,60,60), 1)
    out_dir = DIRS["review_contact_sheets"] / "stability_timelines"
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"track_{int(track['track_id']):04d}_{safe_label(track['export_label'])}_{track['category']}.jpg"
    cv2.imwrite(str(out), canvas)


if CONFIG["SAVE_CONTACT_SHEETS"]:
    important = [t for t in tracks_scored if t["category"] in {"review", "gold", "silver"}]
    for t in sorted(important, key=lambda x: (x["category"] != "review", -x["track_quality_score"]))[:80]:
        make_contact_sheet(t)
    timeline_tracks = sorted(tracks_scored, key=lambda x: (x["category"] == "reject", -x["track_length"], -x["track_quality_score"]))[:60]
    for t in timeline_tracks:
        make_stability_timeline_sheet(t)

def make_split_candidate_sheets(max_tracks=30):
    split_tracks = [t for t in tracks_scored if t.get("split_candidate")]
    out_dir = DIRS["review_contact_sheets"] / "split_candidates"
    out_dir.mkdir(parents=True, exist_ok=True)
    for track in split_tracks[:max_tracks]:
        summary = track.get("split_cluster_summary", {}) or {}
        groups = [set(summary.get("cluster_0_frames", [])), set(summary.get("cluster_1_frames", []))]
        cands = sorted(track.get("candidate_frames", []), key=lambda d: d.get("extracted_frame_idx", 0))
        chosen = []
        for gi, frames in enumerate(groups):
            group_cands = [d for d in cands if d.get("extracted_frame_idx") in frames][:8]
            chosen.extend((gi, d) for d in group_cands)
        if not chosen:
            continue
        tile = 140; cols = 8; rows = math.ceil(len(chosen) / cols)
        canvas = np.full((70 + rows * (tile + 35), cols * tile, 3), 245, dtype=np.uint8)
        title = f"split candidate track {track['track_id']} | {track['export_label']} | sep={summary.get('cluster_separation',0):.2f} var={summary.get('intra_variance',0):.2f}"
        cv2.putText(canvas, title[:150], (8, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (20,20,20), 1)
        cv2.putText(canvas, f"review flags: {track.get('review_flags', [])}"[:150], (8, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (60,60,60), 1)
        for i, (gi, d) in enumerate(chosen):
            crop = cv2.imread(d.get("crop_path", ""))
            if crop is None:
                continue
            crop = cv2.resize(crop, (tile, tile), interpolation=cv2.INTER_AREA)
            r, c = divmod(i, cols); x, y = c * tile, 70 + r * (tile + 35)
            canvas[y:y+tile, x:x+tile] = crop
            color = (60,80,220) if gi == 0 else (40,150,70)
            cv2.rectangle(canvas, (x, y), (x+tile-1, y+tile-1), color, 3)
            cv2.putText(canvas, f"A/B={gi} f{d.get('extracted_frame_idx')}", (x+3, y+tile+15), cv2.FONT_HERSHEY_SIMPLEX, 0.36, color, 1)
            cv2.putText(canvas, str(d.get("raw_label", ""))[:16], (x+3, y+tile+31), cv2.FONT_HERSHEY_SIMPLEX, 0.34, (40,40,40), 1)
        cv2.imwrite(str(out_dir / f"track_{int(track['track_id']):04d}_{safe_label(track['export_label'])}_split.jpg"), canvas)


if CONFIG["SAVE_CONTACT_SHEETS"]:
    make_split_candidate_sheets()

plot_df = track_summary_df.copy()
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()
plot_df["length"].hist(ax=axes[0], bins=20); axes[0].set_title("track lengths")
plot_df["quality"].hist(ax=axes[1], bins=20); axes[1].set_title("quality")
conf_col = "coarse_label_confidence" if "coarse_label_confidence" in plot_df.columns else "label_confidence"
if conf_col in plot_df.columns:
    plot_df[conf_col].hist(ax=axes[2], bins=20)
axes[2].set_title("coarse label confidence")
if "appearance_consistency" in plot_df.columns:
    plot_df["appearance_consistency"].hist(ax=axes[3], bins=20)
axes[3].set_title("appearance consistency")
if "conflict_frame_ratio" in plot_df.columns:
    plot_df["conflict_frame_ratio"].hist(ax=axes[4], bins=20)
axes[4].set_title("conflict frame ratio")
if "split_candidate" in plot_df.columns:
    plot_df["split_candidate"].astype(int).value_counts().sort_index().plot(kind="bar", ax=axes[5])
axes[5].set_title("split candidates")
plt.tight_layout(); plt.show()

if rejected_reasons_counter:
    pd.Series(dict(rejected_reasons_counter.most_common(12))).plot(kind="bar", figsize=(10, 3), title="rejected matches by reason")
    plt.tight_layout(); plt.show()

## 11. Baseline Comparison

In [ ]:

def metrics_from_scored_tracks(tracks_scored, name="tracker_2p5d_antimerge"):
    lengths = [int(t.get("track_length", 0)) for t in tracks_scored]
    short = sum(l < 3 for l in lengths)
    flips = sum(sum(1 for a, b in zip(t.get("raw_label_history", []), t.get("raw_label_history", [])[1:]) if a != b) for t in tracks_scored)
    suspected_merges = sum(bool(t.get("split_candidate")) or t.get("appearance_consistency", 1.0) < CONFIG["MIN_TRACK_APPEARANCE_CONSISTENCY_FOR_EXPORT"] for t in tracks_scored)
    return {
        "method": name,
        "tracks_created": len(tracks_scored),
        "avg_track_length": float(np.mean(lengths)) if lengths else 0,
        "short_tracks": short,
        "estimated_fragmentations": short,
        "raw_label_flips": flips,
        "raw_label_flips_per_track": flips / max(1, len(tracks_scored)),
        "smoothed_label_confidence": float(np.mean([t.get("coarse_label_confidence", t.get("label_confidence", 0)) for t in tracks_scored])) if tracks_scored else 0,
        "suspected_merges": suspected_merges,
        "split_candidates": sum(bool(t.get("split_candidate")) for t in tracks_scored),
        "conflict_zone_frame_ratio": float(np.mean([t.get("conflict_frame_ratio", 0.0) for t in tracks_scored])) if tracks_scored else 0,
        "appearance_consistency": float(np.mean([t.get("appearance_consistency", 0.5) for t in tracks_scored])) if tracks_scored else 0,
        "exportable_tracks": sum(t.get("category") in {"gold", "silver"} for t in tracks_scored),
        "review_tracks": sum(t.get("category") == "review" for t in tracks_scored),
        "gold_tracks": sum(t.get("category") == "gold" for t in tracks_scored),
        "silver_tracks": sum(t.get("category") == "silver" for t in tracks_scored),
    }


comparison_path = DIRS["debug"] / "baseline_vs_2p5d_summary.csv"
previous_rows = []
if comparison_path.exists():
    try:
        prev = pd.read_csv(comparison_path)
        for method in ["baseline_2d", "tracker_2p5d_bytetrack", "tracker_2p5d_without_antimerge"]:
            rows = prev[prev["method"] == method].to_dict(orient="records")
            if rows:
                row = rows[0]
                if row["method"] == "tracker_2p5d_bytetrack":
                    row["method"] = "tracker_2p5d_without_antimerge"
                previous_rows.append(row)
    except Exception as exc:
        print("Could not reuse previous comparison:", exc)

if not any(r.get("method") == "baseline_2d" for r in previous_rows) or not CONFIG.get("REUSE_BASELINE_COMPARISON", True):
    class Baseline2DTracker(TwoStage25DTracker):
        def association_score(self, track, det):
            iou_score = bbox_iou(track.get("bbox", [0,0,0,0]), det.get("bbox", [0,0,0,0]))
            center_s = center_score(track, det)
            label_s = label_score(track, det)
            score = 0.65 * iou_score + 0.30 * center_s + 0.05 * label_s
            comps = {"iou_score": iou_score, "center_score": center_s, "label_score": label_s, "final_score": score}
            return score, comps, True, []

    baseline = Baseline2DTracker()
    for fr in tqdm(enriched_by_frame, desc="baseline 2D tracking"):
        baseline.step(int(fr["extracted_frame_idx"]), fr["detections"])

    def tracker_metrics(tracker, name="baseline_2d"):
        tracks = tracker.all_tracks()
        lengths = [len(set(d["extracted_frame_idx"] for d in t["candidate_frames"])) for t in tracks]
        short = sum(l < 3 for l in lengths)
        flips = sum(sum(1 for a,b in zip(t.get("raw_label_history", []), t.get("raw_label_history", [])[1:]) if a != b) for t in tracks)
        return {
            "method": name,
            "tracks_created": len(tracks),
            "avg_track_length": float(np.mean(lengths)) if lengths else 0,
            "short_tracks": short,
            "estimated_fragmentations": short,
            "raw_label_flips": flips,
            "raw_label_flips_per_track": flips / max(1, len(tracks)),
            "smoothed_label_confidence": 0.0,
            "suspected_merges": 0,
            "split_candidates": 0,
            "conflict_zone_frame_ratio": 0.0,
            "appearance_consistency": 0.0,
            "exportable_tracks": 0,
            "review_tracks": 0,
            "gold_tracks": 0,
            "silver_tracks": 0,
        }
    previous_rows = [r for r in previous_rows if r.get("method") != "baseline_2d"] + [tracker_metrics(baseline)]

comparison_df = pd.DataFrame(previous_rows + [metrics_from_scored_tracks(tracks_scored)])
comparison_df = comparison_df.drop_duplicates(subset=["method"], keep="last")
comparison_df.to_csv(comparison_path, index=False)
display(comparison_df)
print("More raw label flips can happen because longer tracks accumulate more detector label jitter. Use smoothed track-level labels for export.")
print("Longer tracks are useful only if appearance consistency remains high. Anti-merge gates may intentionally split ambiguous tracks to protect dataset quality.")


## 12. Final Feasibility Report Cell

In [ ]:

category_counts = Counter(t["category"] for t in tracks_scored)
export_counts = Counter(rec["export_label"] for rec in selected_crop_records)
export_category_counts = Counter(rec["category"] for rec in selected_crop_records)
base_rows = comparison_df.loc[comparison_df.method == "baseline_2d"] if "method" in comparison_df else pd.DataFrame()
new_rows = comparison_df.loc[comparison_df.method == "tracker_2p5d_antimerge"] if "method" in comparison_df else pd.DataFrame()
base_frag = float(base_rows["estimated_fragmentations"].iloc[0]) if len(base_rows) else 0.0
new_frag = float(new_rows["estimated_fragmentations"].iloc[0]) if len(new_rows) else 0.0
frag_improvement = (base_frag - new_frag) / max(1.0, base_frag)
common_review = Counter(flag for t in tracks_scored for flag in t.get("review_flags", [])).most_common(10)
common_warnings = Counter(flag for t in tracks_scored for flag in t.get("warning_flags", [])).most_common(10)
rejected_reason_counts = Counter(reason for r in rejected_match_rows for reason in r.get("reasons", [])).most_common(10)
app_vals = [t.get("appearance_consistency", 0.5) for t in tracks_scored]
conflict_vals = [t.get("conflict_frame_ratio", 0.0) for t in tracks_scored]

report = {
    "frames_processed": len(frames_metadata),
    "total_dino_sam2_proposals": sum(len(fr["detections"]) for fr in proposals_by_frame),
    "total_enriched_proposals": sum(len(fr["detections"]) for fr in enriched_by_frame),
    "tracks_after_antimerge": len(tracks_scored),
    "average_track_length": float(np.mean([t["track_length"] for t in tracks_scored])) if tracks_scored else 0,
    "appearance_consistency_mean": float(np.mean(app_vals)) if app_vals else 0,
    "appearance_consistency_median": float(np.median(app_vals)) if app_vals else 0,
    "conflict_zone_frame_ratio_mean": float(np.mean(conflict_vals)) if conflict_vals else 0,
    "split_candidates": sum(bool(t.get("split_candidate")) for t in tracks_scored),
    "rejected_matches_by_reason": rejected_reason_counts,
    "track_category_counts": dict(category_counts),
    "export_crop_category_counts": dict(export_category_counts),
    "gold_silver_crop_count": export_category_counts.get("gold", 0) + export_category_counts.get("silver", 0),
    "top_labels_exported": export_counts.most_common(20),
    "fragmentation_improvement_vs_baseline": frag_improvement,
    "average_coarse_label_confidence": float(np.mean([t.get("coarse_label_confidence", t.get("label_confidence", 0)) for t in tracks_scored])) if tracks_scored else 0,
    "average_fine_label_confidence": float(np.mean([t.get("fine_label_confidence", 0) for t in tracks_scored])) if tracks_scored else 0,
    "common_review_reasons": common_review,
    "common_warnings": common_warnings,
}

if report["gold_silver_crop_count"] > 0 and report["appearance_consistency_mean"] >= 0.60 and report["split_candidates"] <= max(3, len(tracks_scored) * 0.25):
    decision = "Promising"
elif report["gold_silver_crop_count"] > 0 or report["appearance_consistency_mean"] >= 0.50:
    decision = "Mixed"
else:
    decision = "Weak"
report["decision"] = decision
(DIRS["debug"] / "feasibility_report.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")

print(json.dumps(report, indent=2, default=str))
print()
print("Decision:", decision)
print()
print("Next tuning suggestions:")
print("- If many nearby objects still merge: increase W_APPEARANCE, MIN_APPEARANCE_IN_CONFLICT, or use DINOv2 instead of CLIP/color hist.")
print("- If tracks fragment too much: lower MIN_APPEARANCE_FOR_MATCH or increase gallery size.")
print("- If split candidates are too many: increase SPLIT_MAX_INTRA_VARIANCE or require higher cluster separation.")
print("- If CLIP confuses similar cookware: use DINOv2 or combine DINOv2 + color histogram.")
print("- If bbox crops are polluted: verify masked crop preprocessing and SAM2 mask quality.")
print("- If conflict frames dominate: reduce export from conflict zones and rely on cleaner representative frames.")


## 13. Constraints and Design Notes

- This notebook is a feasibility experiment, not a GUI or production refactor.
- It uses 2.5D pseudo-3D anchors from monocular depth, not full SLAM, MASt3R, DUSt3R, NeRF, or 3DGS.
- Dynamic object tracks should remain separate from any future static scene map.
- The old DINO+SAM2 crop/mask/proposal output style is preserved as much as possible.
- Track identity and semantic label are intentionally decoupled.
- Gold/silver exports use track-level smoothed/coarse labels, not raw per-frame DINO labels.
- Color histogram appearance is a placeholder; CLIP/DINOv2 embeddings can replace it later.

## Experiment: SAM2 Video Tracking / Mask Propagation

The current DINO+SAM2 pipeline uses SAM2 image prediction frame by frame. SAM2 image prediction does not automatically preserve object masks across time.

This optional notebook branch tests whether the SAM2 video predictor can propagate a clean keyframe mask forward/backward and improve temporal mask consistency. The 2.5D ByteTrack pipeline remains responsible for identity, conflict detection, and safety validation.

Important policy: SAM2 video masks are candidates only. They are not exported as gold/silver and they do not replace good per-frame DINO+SAM2 masks by default.

In [ ]:
CONFIG.update({
    # SAM2 video tracking experiment
    "ENABLE_SAM2_VIDEO_TRACKING_EXPERIMENT": True,

    # SAM2 video model
    "SAM2_VIDEO_CONFIG": "models/sam2/sam2_hiera_t.yaml",
    "SAM2_VIDEO_CHECKPOINT": "models/sam2/sam2_hiera_tiny.pt",
    "SAM2_VIDEO_DEVICE": "cuda",

    # Keyframe selection
    "SAM2_TRACK_SELECTED_TRACK_IDS": None,
    "SAM2_MAX_TRACKS_TO_TEST": 5,
    "SAM2_KEYFRAME_SELECTION": "best_quality",  # best_quality, middle_visible, manual_frame_idx
    "SAM2_MANUAL_KEYFRAMES": {},

    # Propagation window
    "SAM2_PROPAGATE_WINDOW_BEFORE": 25,
    "SAM2_PROPAGATE_WINDOW_AFTER": 50,
    "SAM2_PROPAGATE_FULL_VIDEO": False,

    # Prompt type
    "SAM2_INIT_PROMPT_TYPE": "mask",  # mask, box, points, mask_and_points

    # Clean keyframe requirements
    "SAM2_MIN_KEYFRAME_QUALITY": 0.70,
    "SAM2_REQUIRE_NON_CONFLICT_KEYFRAME": True,
    "SAM2_REQUIRE_VISIBLE_KEYFRAME": True,
    "SAM2_AVOID_SPLIT_CANDIDATE_TRACKS": True,

    # Negative points for re-prompting
    "SAM2_USE_NEGATIVE_POINTS_FROM_OVERLAPS": True,
    "SAM2_NEGATIVE_POINT_MAX": 10,
    "SAM2_POSITIVE_POINT_MAX": 5,

    # Comparison thresholds
    "SAM2_PROP_MIN_MASK_AREA": 100,
    "SAM2_PROP_MAX_AREA_CHANGE_RATIO": 2.5,
    "SAM2_PROP_MIN_IOU_WITH_TRACK_MASK": 0.10,
    "SAM2_PROP_MAX_CENTER_JUMP": 0.20,

    # Safety
    "SAM2_DO_NOT_ACCEPT_PROPAGATED_MASKS_IN_CONFLICT": True,
    "SAM2_DO_NOT_EXPORT_UNVERIFIED_PROPAGATED_MASKS": True,

    # Output
    "SAM2_SAVE_PROPAGATED_MASKS": True,
    "SAM2_SAVE_COMPARISON_VIDEO": True,
    "SAM2_SAVE_DEBUG_JSON": True,
})

CONFIG["SAM2_VIDEO_DEVICE"] = auto_device(CONFIG["SAM2_VIDEO_DEVICE"])
SAM2_VIDEO_DIR = OUTPUT_DIR / "sam2_video_tracking"
SAM2_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
for name in ["masks", "merged_candidates", "keyframes", "visualizations"]:
    (SAM2_VIDEO_DIR / name).mkdir(parents=True, exist_ok=True)

print("SAM2 video experiment config:")
sam2_keys = [k for k in CONFIG if k.startswith("SAM2_") or k == "ENABLE_SAM2_VIDEO_TRACKING_EXPERIMENT"]
print(json.dumps({k: CONFIG[k] for k in sam2_keys}, indent=2, default=str))

### Prepare Video Frames for SAM2 Video Predictor

In [ ]:
def prepare_sam2_video_frames(frames_metadata, output_dir: Path):
    """Create a sequential frame directory for SAM2 video predictor and save frame mapping."""
    sam2_frame_dir = output_dir / "sam2_video_frames_numeric"
    sam2_frame_dir.mkdir(parents=True, exist_ok=True)
    mapping = []
    ordered = sorted(frames_metadata, key=lambda r: int(r["extracted_frame_idx"]))
    for sam2_idx, rec in enumerate(tqdm(ordered, desc="prepare sam2 frames")):
        src = Path(rec["frame_path"])
        dst = sam2_frame_dir / f"{sam2_idx:06d}.jpg"
        if not dst.exists():
            img = cv2.imread(str(src))
            if img is None:
                continue
            cv2.imwrite(str(dst), img)
        mapping.append({
            "sam2_frame_idx": sam2_idx,
            "extracted_frame_idx": int(rec["extracted_frame_idx"]),
            "original_frame_idx": int(rec.get("original_frame_idx", rec["extracted_frame_idx"])),
            "frame_path": str(dst),
            "source_frame_path": str(src),
        })
    (DIRS["debug"] / "sam2_frame_mapping.json").write_text(json.dumps(mapping, indent=2, default=str), encoding="utf-8")
    return sam2_frame_dir, mapping


if CONFIG["ENABLE_SAM2_VIDEO_TRACKING_EXPERIMENT"]:
    sam2_frame_dir, sam2_frame_mapping = prepare_sam2_video_frames(frames_metadata, OUTPUT_DIR)
    extracted_to_sam2 = {int(m["extracted_frame_idx"]): int(m["sam2_frame_idx"]) for m in sam2_frame_mapping}
    sam2_to_extracted = {int(m["sam2_frame_idx"]): int(m["extracted_frame_idx"]) for m in sam2_frame_mapping}
    print("SAM2 frame dir:", sam2_frame_dir)
    print("SAM2 frames:", len(sam2_frame_mapping))

### Select Candidate Tracks and Clean Keyframes

In [ ]:
def mask_area_from_path(mask_path):
    if not mask_path:
        return 0
    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    return int((m > 0).sum()) if m is not None else 0


def visible_candidate_ok(track, cand):
    if CONFIG["SAM2_REQUIRE_VISIBLE_KEYFRAME"] and cand.get("status") not in {None, "visible", "conflict_update"}:
        return False
    if CONFIG["SAM2_REQUIRE_NON_CONFLICT_KEYFRAME"] and cand.get("in_conflict_zone"):
        return False
    if cand.get("quality_score", 0) < CONFIG["SAM2_MIN_KEYFRAME_QUALITY"]:
        return False
    if not cand.get("mask_path") or not Path(cand["mask_path"]).exists():
        return False
    if mask_area_from_path(cand["mask_path"]) < CONFIG["SAM2_PROP_MIN_MASK_AREA"]:
        return False
    return True


def keyframe_score(track, cand):
    return (
        0.35 * float(cand.get("quality_score", 0))
        + 0.25 * float(cand.get("mask_quality_score", 0))
        + 0.20 * float(cand.get("appearance_score_to_track", track.get("appearance_consistency", 0.5)))
        + 0.10 * float(cand.get("conf", 0))
        + 0.10 * (0.0 if cand.get("in_conflict_zone") else 1.0)
    )


def select_keyframe_for_track(track):
    tid = int(track["track_id"])
    manual = CONFIG.get("SAM2_MANUAL_KEYFRAMES", {}) or {}
    if tid in manual or str(tid) in manual:
        frame_idx = int(manual.get(tid, manual.get(str(tid))))
        cands = [c for c in track.get("candidate_frames", []) if int(c.get("extracted_frame_idx", -1)) == frame_idx]
        return cands[0] if cands else None
    candidates = [c for c in track.get("candidate_frames", []) if visible_candidate_ok(track, c)]
    if not candidates:
        candidates = [c for c in track.get("candidate_frames", []) if c.get("mask_path") and Path(c.get("mask_path", "")).exists()]
    if not candidates:
        return None
    if CONFIG["SAM2_KEYFRAME_SELECTION"] == "middle_visible":
        candidates = sorted(candidates, key=lambda c: int(c["extracted_frame_idx"]))
        return candidates[len(candidates) // 2]
    return max(candidates, key=lambda c: keyframe_score(track, c))


def select_tracks_for_sam2(tracks_scored):
    selected_ids = CONFIG.get("SAM2_TRACK_SELECTED_TRACK_IDS")
    if selected_ids:
        wanted = {int(x) for x in selected_ids}
        pool = [t for t in tracks_scored if int(t["track_id"]) in wanted]
    else:
        pool = []
        for t in tracks_scored:
            if int(t.get("track_length", 0)) < 10:
                continue
            if CONFIG["SAM2_AVOID_SPLIT_CANDIDATE_TRACKS"] and t.get("split_candidate"):
                continue
            if t.get("conflict_frame_ratio", 0) > 0.50:
                continue
            if not t.get("candidate_frames"):
                continue
            kf = select_keyframe_for_track(t)
            if kf is None:
                continue
            # Prefer informative tracks: conflicts, misses, label jitter, or important object classes.
            informative = (
                t.get("missing_count", 0) > 0
                or t.get("occluded_count", 0) > 0
                or t.get("conflict_frame_ratio", 0) > 0.05
                or t.get("raw_label_entropy", 0) > 0.6
                or t.get("export_label") in {"hand", "cookware", "utensil", "container", "sink", "bottle"}
            )
            if informative:
                pool.append(t)
    scored = []
    for t in pool:
        kf = select_keyframe_for_track(t)
        if kf is None:
            continue
        reason = []
        if t.get("missing_count", 0): reason.append("has_missing")
        if t.get("occluded_count", 0): reason.append("has_occlusion")
        if t.get("conflict_frame_ratio", 0) > 0.05: reason.append("conflict_test")
        if t.get("raw_label_entropy", 0) > 0.6: reason.append("label_jitter")
        reason.append("clean_keyframe")
        scored.append((t.get("track_quality_score", 0), t, kf, ",".join(reason)))
    scored.sort(key=lambda x: x[0], reverse=True)
    rows, selected = [], []
    for _, t, kf, reason in scored[:CONFIG["SAM2_MAX_TRACKS_TO_TEST"]]:
        tid = int(t["track_id"])
        rows.append({
            "track_id": tid,
            "export_label": t.get("export_label"),
            "length": t.get("track_length"),
            "quality": t.get("track_quality_score"),
            "conflict_frame_ratio": t.get("conflict_frame_ratio"),
            "keyframe_idx": int(kf["extracted_frame_idx"]),
            "keyframe_quality": kf.get("quality_score"),
            "reason_selected": reason,
        })
        selected.append({"track": t, "keyframe": kf, "reason": reason})
    df = pd.DataFrame(rows)
    df.to_csv(DIRS["debug"] / "sam2_selected_tracks.csv", index=False)
    display(df)
    return selected, df


if CONFIG["ENABLE_SAM2_VIDEO_TRACKING_EXPERIMENT"]:
    sam2_selected_tracks, sam2_selected_df = select_tracks_for_sam2(tracks_scored)
    print("selected tracks:", len(sam2_selected_tracks))

In [ ]:
def load_binary_mask(path):
    m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        return None
    return (m > 0).astype(np.uint8)


def sample_points_from_mask(mask, max_points=5):
    mask = (mask > 0).astype(np.uint8)
    core = cv2.erode(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)), iterations=1)
    if core.sum() == 0:
        core = mask
    ys, xs = np.where(core > 0)
    if len(xs) == 0:
        return np.empty((0, 2), dtype=np.float32)
    pts = []
    cx, cy = float(xs.mean()), float(ys.mean())
    pts.append([cx, cy])
    if max_points > 1:
        rng = np.random.default_rng(42)
        idxs = rng.choice(len(xs), size=min(max_points - 1, len(xs)), replace=False)
        pts.extend([[float(xs[i]), float(ys[i])] for i in idxs])
    return np.asarray(pts[:max_points], dtype=np.float32)


def sample_negative_points_from_overlaps(target_mask, keyframe_det, frame_detections, max_points=10):
    if not CONFIG["SAM2_USE_NEGATIVE_POINTS_FROM_OVERLAPS"]:
        return np.empty((0, 2), dtype=np.float32)
    neg = []
    target = target_mask.astype(bool)
    for d in frame_detections:
        if d is keyframe_det or d.get("mask_path") == keyframe_det.get("mask_path"):
            continue
        m = load_binary_mask(d.get("mask_path", ""))
        if m is None or m.shape != target_mask.shape:
            continue
        overlap = (m.astype(bool) & target).sum() / max(1, m.sum())
        if overlap > 0.05:
            ys, xs = np.where(m.astype(bool) & ~target)
            if len(xs):
                take = min(2, len(xs), max_points - len(neg))
                for i in np.linspace(0, len(xs) - 1, take).astype(int):
                    neg.append([float(xs[i]), float(ys[i])])
        if len(neg) >= max_points:
            break
    # Add local ring negatives just outside target.
    if len(neg) < max_points:
        ring = cv2.dilate(target_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)), iterations=1).astype(bool) & ~target
        ys, xs = np.where(ring)
        if len(xs):
            take = min(max_points - len(neg), len(xs))
            for i in np.linspace(0, len(xs) - 1, take).astype(int):
                neg.append([float(xs[i]), float(ys[i])])
    return np.asarray(neg[:max_points], dtype=np.float32)


def find_frame_record(frame_idx):
    for fr in enriched_by_frame:
        if int(fr["extracted_frame_idx"]) == int(frame_idx):
            return fr
    return None


keyframe_payloads = []
for item in sam2_selected_tracks if CONFIG["ENABLE_SAM2_VIDEO_TRACKING_EXPERIMENT"] else []:
    track, kf = item["track"], item["keyframe"]
    mask = load_binary_mask(kf.get("mask_path", ""))
    if mask is None:
        continue
    frame_rec = find_frame_record(int(kf["extracted_frame_idx"]))
    frame_dets = frame_rec.get("detections", []) if frame_rec else []
    pos = sample_points_from_mask(mask, CONFIG["SAM2_POSITIVE_POINT_MAX"])
    neg = sample_negative_points_from_overlaps(mask, kf, frame_dets, CONFIG["SAM2_NEGATIVE_POINT_MAX"])
    keyframe_payloads.append({
        "track_id": int(track["track_id"]),
        "label": track.get("export_label"),
        "keyframe_idx": int(kf["extracted_frame_idx"]),
        "sam2_keyframe_idx": extracted_to_sam2.get(int(kf["extracted_frame_idx"])),
        "mask_path": kf.get("mask_path"),
        "bbox": kf.get("bbox"),
        "positive_points": pos.tolist(),
        "negative_points": neg.tolist(),
        "quality": kf.get("quality_score"),
        "reason": item["reason"],
    })
    # Keyframe visualization.
    img = cv2.imread(str(Path(kf["frame_path"] if "frame_path" in kf else frames_metadata[int(kf["extracted_frame_idx"])]["frame_path"])))
    if img is not None:
        overlay = img.copy()
        overlay[mask.astype(bool)] = (0.55 * overlay[mask.astype(bool)] + 0.45 * np.array([40, 220, 80])).astype(np.uint8)
        x1, y1, x2, y2 = [int(v) for v in kf.get("bbox", [0, 0, 0, 0])]
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(overlay, f"track {track['track_id']} {track.get('export_label')} q={kf.get('quality_score',0):.2f}", (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,255,255), 2)
        cv2.imwrite(str(SAM2_VIDEO_DIR / "keyframes" / f"track_{int(track['track_id']):04d}_keyframe_{int(kf['extracted_frame_idx']):06d}.jpg"), overlay)

(DIRS["debug"] / "sam2_keyframes.json").write_text(json.dumps(keyframe_payloads, indent=2, default=str), encoding="utf-8")
print("keyframes:", len(keyframe_payloads))
display(pd.DataFrame(keyframe_payloads))

### Load SAM2 Video Predictor

In [ ]:
def resolve_sam2_video_paths():
    cfg = CONFIG.get("SAM2_VIDEO_CONFIG") or CONFIG.get("SAM2_CONFIG") or "models/sam2/sam2_hiera_t.yaml"
    ckpt = CONFIG.get("SAM2_VIDEO_CHECKPOINT") or CONFIG.get("SAM2_CHECKPOINT") or "models/sam2/sam2_hiera_tiny.pt"
    cfg_path = repo_path(cfg)
    ckpt_path = repo_path(ckpt)
    return cfg_path, ckpt_path


def load_sam2_video_predictor(config):
    if not config["ENABLE_SAM2_VIDEO_TRACKING_EXPERIMENT"]:
        return None, "disabled"
    cfg_path, ckpt_path = resolve_sam2_video_paths()
    if cfg_path is None or ckpt_path is None or not ckpt_path.exists():
        return None, f"SAM2 video config/checkpoint missing. config={cfg_path}, checkpoint={ckpt_path}"
    try:
        from sam2.build_sam import build_sam2_video_predictor
        predictor = build_sam2_video_predictor(str(cfg_path), str(ckpt_path), device=config["SAM2_VIDEO_DEVICE"])
        return predictor, "loaded: sam2.build_sam.build_sam2_video_predictor"
    except Exception as exc1:
        try:
            from sam2.build_sam2 import build_sam2_video_predictor
            predictor = build_sam2_video_predictor(str(cfg_path), str(ckpt_path), device=config["SAM2_VIDEO_DEVICE"])
            return predictor, "loaded: sam2.build_sam2.build_sam2_video_predictor"
        except Exception as exc2:
            return None, f"SAM2 video predictor unavailable: {type(exc1).__name__}: {exc1}; fallback error: {type(exc2).__name__}: {exc2}"


sam2_video_predictor, sam2_video_load_status = load_sam2_video_predictor(CONFIG)
print("SAM2 video predictor:", sam2_video_load_status)

### Run SAM2 Video Propagation

In [ ]:
def bbox_xyxy_to_numpy(box):
    return np.asarray(box, dtype=np.float32).reshape(1, 4)


def extract_mask_from_logits(mask_logits):
    if hasattr(mask_logits, "detach"):
        arr = mask_logits.detach().float().cpu().numpy()
    else:
        arr = np.asarray(mask_logits)
    arr = np.squeeze(arr)
    return (arr > 0).astype(np.uint8)


def try_add_prompt(predictor, state, payload, mask):
    obj_id = int(payload["track_id"])
    frame_idx = int(payload["sam2_keyframe_idx"])
    prompt_type = CONFIG["SAM2_INIT_PROMPT_TYPE"]
    points = np.asarray(payload.get("positive_points", []), dtype=np.float32)
    neg = np.asarray(payload.get("negative_points", []), dtype=np.float32)
    if len(points) and len(neg):
        all_points = np.vstack([points, neg])
        labels = np.asarray([1] * len(points) + [0] * len(neg), dtype=np.int32)
    elif len(points):
        all_points = points
        labels = np.asarray([1] * len(points), dtype=np.int32)
    else:
        all_points, labels = None, None
    # SAM2 APIs differ across versions. Try mask prompt first, then points/box.
    if prompt_type in {"mask", "mask_and_points"} and hasattr(predictor, "add_new_mask"):
        return predictor.add_new_mask(inference_state=state, frame_idx=frame_idx, obj_id=obj_id, mask=mask.astype(bool)), "mask"
    if hasattr(predictor, "add_new_points_or_box") and all_points is not None:
        kwargs = dict(inference_state=state, frame_idx=frame_idx, obj_id=obj_id, points=all_points, labels=labels)
        if prompt_type in {"box", "mask_and_points"} and payload.get("bbox") is not None:
            kwargs["box"] = np.asarray(payload["bbox"], dtype=np.float32)
        return predictor.add_new_points_or_box(**kwargs), "points_or_box"
    if hasattr(predictor, "add_new_points_or_box") and payload.get("bbox") is not None:
        return predictor.add_new_points_or_box(inference_state=state, frame_idx=frame_idx, obj_id=obj_id, box=np.asarray(payload["bbox"], dtype=np.float32)), "box"
    raise RuntimeError("No supported SAM2 prompt API found on predictor")


def propagate_one_track(predictor, payload):
    tid = int(payload["track_id"])
    kf_ex = int(payload["keyframe_idx"])
    kf_sam2 = int(payload["sam2_keyframe_idx"])
    if CONFIG["SAM2_PROPAGATE_FULL_VIDEO"]:
        start_sam2, end_sam2 = 0, len(sam2_frame_mapping) - 1
    else:
        start_ex = max(0, kf_ex - CONFIG["SAM2_PROPAGATE_WINDOW_BEFORE"])
        end_ex = min(max(extracted_to_sam2), kf_ex + CONFIG["SAM2_PROPAGATE_WINDOW_AFTER"])
        start_sam2 = extracted_to_sam2.get(start_ex, max(0, kf_sam2 - CONFIG["SAM2_PROPAGATE_WINDOW_BEFORE"]))
        end_sam2 = extracted_to_sam2.get(end_ex, min(len(sam2_frame_mapping) - 1, kf_sam2 + CONFIG["SAM2_PROPAGATE_WINDOW_AFTER"]))
    out_dir = SAM2_VIDEO_DIR / "masks" / f"track_{tid:04d}"
    out_dir.mkdir(parents=True, exist_ok=True)
    key_mask = load_binary_mask(payload["mask_path"])
    meta = {"track_id": tid, "keyframe_idx": kf_ex, "sam2_keyframe_idx": kf_sam2, "propagation_start": start_sam2, "propagation_end": end_sam2, "frames": [], "error": ""}
    try:
        state = predictor.init_state(video_path=str(sam2_frame_dir))
        _, used_prompt = try_add_prompt(predictor, state, payload, key_mask)
        meta["prompt_type_used"] = used_prompt
        # Forward propagation.
        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(state, start_frame_idx=kf_sam2):
            if out_frame_idx < start_sam2 or out_frame_idx > end_sam2:
                continue
            for obj_i, obj_id in enumerate(out_obj_ids):
                if int(obj_id) != tid:
                    continue
                mask = extract_mask_from_logits(out_mask_logits[obj_i])
                ex_idx = sam2_to_extracted.get(int(out_frame_idx), int(out_frame_idx))
                mpath = out_dir / f"frame_{ex_idx:06d}.png"
                cv2.imwrite(str(mpath), mask * 255)
                meta["frames"].append({"sam2_frame_idx": int(out_frame_idx), "extracted_frame_idx": int(ex_idx), "mask_path": str(mpath), "area": int(mask.sum())})
        # Some SAM2 versions support reverse propagation.
        try:
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(state, start_frame_idx=kf_sam2, reverse=True):
                if out_frame_idx < start_sam2 or out_frame_idx > end_sam2:
                    continue
                for obj_i, obj_id in enumerate(out_obj_ids):
                    if int(obj_id) != tid:
                        continue
                    mask = extract_mask_from_logits(out_mask_logits[obj_i])
                    ex_idx = sam2_to_extracted.get(int(out_frame_idx), int(out_frame_idx))
                    mpath = out_dir / f"frame_{ex_idx:06d}.png"
                    cv2.imwrite(str(mpath), mask * 255)
                    meta["frames"].append({"sam2_frame_idx": int(out_frame_idx), "extracted_frame_idx": int(ex_idx), "mask_path": str(mpath), "area": int(mask.sum())})
        except TypeError:
            pass
        # Deduplicate frames.
        by_frame = {int(r["extracted_frame_idx"]): r for r in meta["frames"]}
        meta["frames"] = [by_frame[k] for k in sorted(by_frame)]
    except Exception as exc:
        meta["error"] = f"{type(exc).__name__}: {exc}"
    (DIRS["debug"] / f"sam2_propagation_track_{tid}.json").write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")
    return meta


sam2_propagation_results = []
if CONFIG["ENABLE_SAM2_VIDEO_TRACKING_EXPERIMENT"] and sam2_video_predictor is not None:
    for payload in keyframe_payloads:
        sam2_propagation_results.append(propagate_one_track(sam2_video_predictor, payload))
else:
    print("Skipping SAM2 video propagation:", sam2_video_load_status)

print("propagation result tracks:", len(sam2_propagation_results))
for r in sam2_propagation_results:
    print(r["track_id"], "frames", len(r.get("frames", [])), "error", r.get("error", ""))

### Compare SAM2 Propagated Masks with Existing Per-frame Masks

In [ ]:
def bbox_from_mask(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    return [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def mask_iou(a, b):
    aa, bb = a.astype(bool), b.astype(bool)
    return float((aa & bb).sum() / max(1, (aa | bb).sum()))


def mask_center(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    return np.asarray([float(xs.mean()), float(ys.mean())], dtype=np.float32)


track_by_id = {int(t["track_id"]): t for t in tracks_scored}


def candidate_for_frame(track, frame_idx):
    cands = [c for c in track.get("candidate_frames", []) if int(c.get("extracted_frame_idx", -1)) == int(frame_idx)]
    return cands[0] if cands else None


def evaluate_propagated_mask_safety(track, frame_idx, prop_mask, existing_cand):
    area = int(prop_mask.sum())
    if area < CONFIG["SAM2_PROP_MIN_MASK_AREA"]:
        return "reject_area_jump", ["too_small"]
    key_area = mask_area_from_path(select_keyframe_for_track(track).get("mask_path"))
    area_ratio = area / max(1, key_area)
    reasons = []
    if area_ratio > CONFIG["SAM2_PROP_MAX_AREA_CHANGE_RATIO"] or area_ratio < 1.0 / CONFIG["SAM2_PROP_MAX_AREA_CHANGE_RATIO"]:
        reasons.append("area_jump")
    if existing_cand is not None and existing_cand.get("mask_path"):
        orig = load_binary_mask(existing_cand["mask_path"])
        if orig is not None:
            miou = mask_iou(prop_mask, orig)
            if miou < CONFIG["SAM2_PROP_MIN_IOU_WITH_TRACK_MASK"]:
                reasons.append("low_iou_with_track_mask")
            c1, c2 = mask_center(prop_mask), mask_center(orig)
            if c1 is not None and c2 is not None:
                h, w = prop_mask.shape[:2]
                center_jump = float(np.linalg.norm(c1 - c2) / max(w, h))
                if center_jump > CONFIG["SAM2_PROP_MAX_CENTER_JUMP"]:
                    reasons.append("center_jump")
    if existing_cand is not None and existing_cand.get("in_conflict_zone") and CONFIG["SAM2_DO_NOT_ACCEPT_PROPAGATED_MASKS_IN_CONFLICT"]:
        reasons.append("conflict_zone")
    if "area_jump" in reasons:
        return "reject_area_jump", reasons
    if "center_jump" in reasons or "low_iou_with_track_mask" in reasons:
        return "reject_drift", reasons
    if "conflict_zone" in reasons:
        return "review_candidate", reasons
    if existing_cand is None:
        return "recovery_candidate", ["missing_original_track_mask"]
    return "accept_candidate", reasons


comparison_rows = []
for result in sam2_propagation_results:
    tid = int(result["track_id"])
    track = track_by_id.get(tid)
    if track is None:
        continue
    prev_area = None
    for fr in result.get("frames", []):
        frame_idx = int(fr["extracted_frame_idx"])
        prop = load_binary_mask(fr["mask_path"])
        if prop is None:
            continue
        existing = candidate_for_frame(track, frame_idx)
        orig_iou = np.nan
        center_dist = np.nan
        if existing is not None and existing.get("mask_path"):
            orig = load_binary_mask(existing["mask_path"])
            if orig is not None:
                orig_iou = mask_iou(prop, orig)
                c1, c2 = mask_center(prop), mask_center(orig)
                if c1 is not None and c2 is not None:
                    center_dist = float(np.linalg.norm(c1 - c2) / max(prop.shape[:2]))
        safety, reasons = evaluate_propagated_mask_safety(track, frame_idx, prop, existing)
        area = int(prop.sum())
        comparison_rows.append({
            "track_id": tid,
            "label": track.get("export_label"),
            "frame_idx": frame_idx,
            "prop_mask_path": fr["mask_path"],
            "has_original_track_mask": existing is not None,
            "iou_with_original": orig_iou,
            "center_distance": center_dist,
            "prop_area": area,
            "area_ratio_prev": area / max(1, prev_area) if prev_area else np.nan,
            "safety_class": safety,
            "safety_reasons": ";".join(reasons),
            "possible_recovery": safety == "recovery_candidate",
            "conflict_zone": bool(existing.get("in_conflict_zone")) if existing else False,
        })
        prev_area = area

sam2_comparison_df = pd.DataFrame(comparison_rows)
sam2_comparison_df.to_csv(DIRS["debug"] / "sam2_vs_perframe_comparison.csv", index=False)
display(sam2_comparison_df.head(30))

### Optional Merge Strategy and Visualizations

In [ ]:
def make_merged_candidates():
    rows = []
    merged_dir = SAM2_VIDEO_DIR / "merged_candidates"
    merged_dir.mkdir(parents=True, exist_ok=True)
    if sam2_comparison_df.empty:
        return pd.DataFrame()
    for _, row in sam2_comparison_df.iterrows():
        tid, frame_idx = int(row["track_id"]), int(row["frame_idx"])
        track = track_by_id.get(tid)
        existing = candidate_for_frame(track, frame_idx) if track else None
        chosen = "original"
        chosen_path = existing.get("mask_path") if existing else ""
        if (existing is None or existing.get("quality_score", 0) < 0.45) and row["safety_class"] in {"accept_candidate", "recovery_candidate"}:
            chosen = "sam2_propagated_recovery"
            chosen_path = row["prop_mask_path"]
            out = merged_dir / f"track_{tid:04d}_frame_{frame_idx:06d}.png"
            m = load_binary_mask(chosen_path)
            if m is not None:
                cv2.imwrite(str(out), m * 255)
                chosen_path = str(out)
        rows.append({"track_id": tid, "frame_idx": frame_idx, "selected_mask_source": chosen, "mask_path": chosen_path, "safety_class": row["safety_class"]})
    df = pd.DataFrame(rows)
    df.to_csv(DIRS["debug"] / "sam2_merged_candidate_summary.csv", index=False)
    return df


sam2_merged_df = make_merged_candidates()
display(sam2_merged_df.head(30))


def overlay_mask_bgr(img, mask, color, alpha=0.45):
    out = img.copy()
    if mask is None:
        return out
    sel = mask.astype(bool)
    out[sel] = (1 - alpha) * out[sel] + alpha * np.asarray(color, dtype=np.uint8)
    return out.astype(np.uint8)


def frame_path_for_idx(frame_idx):
    rec = frames_metadata[int(frame_idx)] if int(frame_idx) < len(frames_metadata) else None
    if rec and int(rec.get("extracted_frame_idx", frame_idx)) == int(frame_idx):
        return rec["frame_path"]
    for rec in frames_metadata:
        if int(rec["extracted_frame_idx"]) == int(frame_idx):
            return rec["frame_path"]
    return None


def make_track_contact_sheet(track_id, max_tiles=18):
    rows = sam2_comparison_df[sam2_comparison_df["track_id"] == track_id].sort_values("frame_idx")
    if rows.empty:
        return None
    if len(rows) > max_tiles:
        idxs = np.linspace(0, len(rows) - 1, max_tiles).round().astype(int)
        rows = rows.iloc[idxs]
    tile_w, tile_h = 220, 165
    canvas = np.full((len(rows) * tile_h, tile_w * 3, 3), 245, dtype=np.uint8)
    track = track_by_id[track_id]
    for r_i, (_, row) in enumerate(rows.iterrows()):
        frame_idx = int(row["frame_idx"])
        img_path = frame_path_for_idx(frame_idx)
        img = cv2.imread(str(img_path)) if img_path else None
        if img is None:
            continue
        img = cv2.resize(img, (tile_w, tile_h))
        existing = candidate_for_frame(track, frame_idx)
        orig_mask = load_binary_mask(existing["mask_path"]) if existing and existing.get("mask_path") else None
        prop_mask = load_binary_mask(row["prop_mask_path"])
        if orig_mask is not None:
            orig_mask = cv2.resize(orig_mask, (tile_w, tile_h), interpolation=cv2.INTER_NEAREST)
        if prop_mask is not None:
            prop_mask = cv2.resize(prop_mask, (tile_w, tile_h), interpolation=cv2.INTER_NEAREST)
        left = overlay_mask_bgr(img, orig_mask, (0, 220, 80))
        mid = overlay_mask_bgr(img, prop_mask, (220, 80, 30))
        diff = img.copy()
        if orig_mask is not None and prop_mask is not None:
            kept = orig_mask.astype(bool) & prop_mask.astype(bool)
            only_orig = orig_mask.astype(bool) & ~prop_mask.astype(bool)
            only_prop = prop_mask.astype(bool) & ~orig_mask.astype(bool)
            diff[kept] = (50, 210, 80)
            diff[only_orig] = (40, 40, 230)
            diff[only_prop] = (230, 140, 40)
        y = r_i * tile_h
        canvas[y:y+tile_h, 0:tile_w] = left
        canvas[y:y+tile_h, tile_w:tile_w*2] = mid
        canvas[y:y+tile_h, tile_w*2:tile_w*3] = diff
        txt = f"f{frame_idx} {row['safety_class']} iou={row.get('iou_with_original', np.nan):.2f}"
        cv2.putText(canvas, txt[:70], (5, y + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (10,10,10), 1)
    out = SAM2_VIDEO_DIR / "visualizations" / f"sam2_video_track_{track_id:04d}_contact_sheet.jpg"
    cv2.imwrite(str(out), canvas)
    return out


contact_paths = []
for tid in sorted(sam2_comparison_df["track_id"].unique()) if not sam2_comparison_df.empty else []:
    p = make_track_contact_sheet(int(tid))
    if p:
        contact_paths.append(p)
print("contact sheets:", contact_paths)

### Quantitative Summary and Interpretation

In [ ]:
summary_rows = []
for item in keyframe_payloads:
    tid = int(item["track_id"])
    rows = sam2_comparison_df[sam2_comparison_df["track_id"] == tid] if not sam2_comparison_df.empty else pd.DataFrame()
    track = track_by_id.get(tid, {})
    if rows.empty:
        summary_rows.append({"track_id": tid, "label": item.get("label"), "keyframe": item.get("keyframe_idx"), "recommendation": "not_run_or_no_masks"})
        continue
    ious = rows["iou_with_original"].dropna()
    areas = rows["prop_area"].astype(float)
    cls = rows["safety_class"].value_counts().to_dict()
    summary_rows.append({
        "track_id": tid,
        "label": item.get("label"),
        "keyframe": item.get("keyframe_idx"),
        "propagation_window_length": int(len(rows)),
        "frames_with_original_track_mask": int(rows["has_original_track_mask"].sum()),
        "frames_with_sam2_propagated_mask": int(len(rows)),
        "mean_iou_with_original": float(ious.mean()) if len(ious) else np.nan,
        "median_iou_with_original": float(ious.median()) if len(ious) else np.nan,
        "mask_area_stability_sam2": float(1.0 / (1.0 + areas.std() / max(1.0, areas.mean()))) if len(areas) else np.nan,
        "possible_recovery_count": int(rows["possible_recovery"].sum()),
        "drift_reject_count": int((rows["safety_class"] == "reject_drift").sum()),
        "conflict_reject_count": int((rows["safety_class"] == "reject_conflict").sum()) if "reject_conflict" in rows["safety_class"].values else 0,
        "accept_candidate_count": int((rows["safety_class"] == "accept_candidate").sum()),
        "review_candidate_count": int((rows["safety_class"] == "review_candidate").sum()),
        "reject_area_jump_count": int((rows["safety_class"] == "reject_area_jump").sum()),
        "recommendation": "review/recovery_candidate_branch",
    })

sam2_summary_df = pd.DataFrame(summary_rows)
sam2_summary_df.to_csv(DIRS["debug"] / "sam2_video_tracking_summary.csv", index=False)
display(sam2_summary_df)

if sam2_summary_df.empty or (sam2_summary_df.get("frames_with_sam2_propagated_mask", pd.Series(dtype=int)).fillna(0).sum() == 0):
    conclusion = "SAM2 video propagation did not run. Check SAM2 video config/checkpoint/API."
else:
    drift = sam2_summary_df.get("drift_reject_count", pd.Series(dtype=float)).fillna(0).sum()
    recover = sam2_summary_df.get("possible_recovery_count", pd.Series(dtype=float)).fillna(0).sum()
    accept = sam2_summary_df.get("accept_candidate_count", pd.Series(dtype=float)).fillna(0).sum()
    review = sam2_summary_df.get("review_candidate_count", pd.Series(dtype=float)).fillna(0).sum()
    if accept > 0 and recover > 0 and drift < accept:
        conclusion = "SAM2 video propagation is promising."
    elif recover > 0 or review > 0:
        conclusion = "SAM2 video propagation is useful only as review/recovery."
    else:
        conclusion = "SAM2 video propagation is not safe for automatic export yet."

print(conclusion)
print()
print("Recommended next steps:")
print("- Use SAM2 video only on high-quality keyframes.")
print("- Do not initialize from over-merged masks.")
print("- Use negative points from overlapping object masks.")
print("- Use 2.5D tracker to validate propagated masks.")
print("- Use propagated masks as recovery/review candidates, not gold/silver by default.")
print("- Prefer short propagation windows around reliable keyframes.")
print("- Use appearance embeddings and depth consistency to detect drift.")